In [ ]:
# =============================📖 BEGIN Header 📝=============================
# SIPREV-Mulher/MS
# Sistema Inteligente de Predição e Mapeamento da
# Violência contra a Mulher em Mato Grosso do Sul
# 📝 Large Language Model (LLM)
# 🧠 Neural networks, Machine learning, and Deep machine learning
# =============================================================================
# Disciplina : 📚 Tópicos Interdisciplinares III
# Curso      : Ciência dos Dados — UFMS Digital
# Semestre   : 2026.1
# Autor      : VIANA
# Fonte dados: SINAN/DATASUS — OpenScienceTechnology/GitHub
# =============================📝 END Footer 📄=============================

# 🛡️ SIPREV-Mulher/MS
## Sistema Inteligente de Predição e Mapeamento da Violência contra a Mulher em Mato Grosso do Sul

**Disciplina:** Tópicos Interdisciplinares III | **Curso:** Ciência dos Dados — UFMS Digital | **Semestre:** 2026.1

---
### Sumário
0. Configuração do Ambiente e Instalação de Dependências
1. Carregamento e Inspeção Inicial dos Dados
2. Limpeza e Pré-processamento
3. Análise Exploratória Descritiva (AED)
4. Análise Geoespacial
5. Clusterização Municipal
6. Modelos Preditivos Clássicos (ML)
7. Deep Learning (LSTM / GRU / MLP)
8. Explicabilidade com SHAP e LIME
9. Dashboard Streamlit (app.py)
10. Conclusões e Recomendações de Política Pública


============================================================
SEÇÃO 0 — CONFIGURAÇÃO DO AMBIENTE E INSTALAÇÃO
============================================================

## Seção 0 — Configuração do Ambiente e Instalação de Dependências

Execute esta célula uma única vez para instalar todas as bibliotecas necessárias.
O bloco é compatível com **Google Colab**, **Jupyter Notebook** e ambientes locais.

In [ ]:
# ------------------------------------------------------------------
# 0.1  Instalação de dependências
# ------------------------------------------------------------------
import subprocess, sys

PACKAGES = [
    # Manipulação de dados
    "pandas", "numpy", "polars", "dask[dataframe]", "duckdb",
    "pyarrow", "fastparquet",
    # Qualidade e validação
    "pandera", "pydantic", "great_expectations",
    "missingno", "unidecode", "rapidfuzz",
    # Análise estatística
    "scipy", "statsmodels", "ydata-profiling",
    # Geoespacial
    "geopandas", "shapely", "folium", "contextily",
    "pydeck", "geopy",
    # Machine Learning
    "scikit-learn", "xgboost", "lightgbm",
    "imbalanced-learn", "hdbscan", "optuna",
    # Séries temporais
    "prophet", "pmdarima",
    # Deep Learning
    "tensorflow", "torch", "pytorch-lightning",
    "torchmetrics",
    # Explicabilidade
    "shap", "lime", "eli5",
    # MLOps
    "mlflow", "dvc", "loguru", "hydra-core",
    "joblib", "onnx", "onnxruntime",
    # Visualização e Dashboard
    "matplotlib", "seaborn", "plotly", "altair",
    "streamlit",
    # Utilitários
    "requests", "tqdm", "colorama",
]

def install(pkg):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )

print("Instalando dependências — aguarde...")
for p in PACKAGES:
    try:
        install(p)
        print(f"  ✅ {p}")
    except Exception as e:
        print(f"  ⚠️  {p} — {e}")
print("\n🎉 Ambiente configurado com sucesso!")

In [ ]:
# ------------------------------------------------------------------
# 0.2  Importações globais
# ------------------------------------------------------------------
# — Dados e numérico —
import pandas as pd
import numpy as np
import polars as pl
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

# — Qualidade de dados —
import pandera as pda
import missingno as msno
from unidecode import unidecode
from rapidfuzz import process as rfprocess

# — Estatística —
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# — Geoespacial —
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap, MarkerCluster
import contextily as ctx
import pydeck as pdk
from geopy.geocoders import Nominatim

# — Machine Learning —
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    train_test_split, cross_val_score, TimeSeriesSplit
)
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    IsolationForest
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, mean_absolute_error,
    mean_squared_error, r2_score,
    classification_report, roc_auc_score, f1_score,
    confusion_matrix
)
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import hdbscan
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# — Séries temporais —
from prophet import Prophet
from pmdarima import auto_arima

# — Deep Learning —
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# — Explicabilidade —
import shap
import lime
import lime.lime_tabular

# — MLOps —
import mlflow
import mlflow.sklearn
from loguru import logger
import joblib

# — Visualização —
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import altair as alt

# — Utilitários —
import os, re, warnings, requests
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Todas as importações realizadas com sucesso.")
print(f"   pandas  {pd.__version__} | numpy {np.__version__} | sklearn {__import__('sklearn').__version__}")
print(f"   TensorFlow {tf.__version__} | XGBoost {xgb.__version__} | LightGBM {lgb.__version__}")

In [ ]:
# ------------------------------------------------------------------
# 0.3  Constantes e configurações do projeto
# ------------------------------------------------------------------
# Código IBGE de Mato Grosso do Sul
UF_MS  = 50
SG_MS  = "50"          # código string em SG_UF_NOT

# URLs dos CSVs no GitHub (2009–2025)
BASE_URL = (
    "https://media.githubusercontent.com/media/"
    "OpenScienceTechnology/Microdados_SINAN-DATASUS/"
    "refs/heads/main/Data/CSV/"
)
ANOS = list(range(2009, 2026))
CSV_URLS = {ano: f"{BASE_URL}VIOLBR{str(ano)[2:]}.csv" for ano in ANOS}

# Diretórios de trabalho
DATA_DIR    = Path("data/raw")
PROC_DIR    = Path("data/processed")
OUTPUT_DIR  = Path("outputs")
MODEL_DIR   = Path("models")
LOG_DIR     = Path("logs")

for d in [DATA_DIR, PROC_DIR, OUTPUT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Logger
logger.add(LOG_DIR / "siprev_{time}.log", rotation="10 MB", level="INFO")
logger.info("SIPREV-Mulher/MS — Sessão iniciada")

# Dicionários de decodificação SINAN
DICT_RACA = {1: "Branca", 2: "Preta", 3: "Amarela",
             4: "Parda", 5: "Indígena", 9: "Ignorado"}

DICT_ESCOL = {
    0: "Analfabeto", 1: "1ª-4ª série incomp.",
    2: "4ª série comp.", 3: "5ª-8ª série incomp.",
    4: "Fund. completo", 5: "Médio incomp.",
    6: "Médio completo", 7: "Superior incomp.",
    8: "Superior completo", 9: "Ignorado", 10: "Não se aplica"
}

DICT_LOCAL = {
    1: "Residência", 2: "Hab. coletiva",
    3: "Escola", 4: "Local de prática esportiva",
    5: "Bar / Similar", 6: "Via pública",
    7: "Comércio / Serviços", 8: "Indústria",
    9: "Outro", 99: "Ignorado"
}

DICT_SEXO  = {"F": "Feminino", "M": "Masculino", "I": "Ignorado"}

DICT_EVOLUCAO = {
    1: "Cura/Alta", 2: "Óbito por violência",
    3: "Óbito por outra causa", 4: "Encerramento",
    9: "Ignorado"
}

COLS_VIOL = [
    "VIOL_FISIC", "VIOL_PSICO", "VIOL_TORT",
    "VIOL_SEXU", "VIOL_TRAF", "VIOL_FINAN",
    "VIOL_NEGLI", "VIOL_INFAN", "VIOL_LEGAL", "VIOL_OUTR"
]
LABELS_VIOL = [
    "Física", "Psicológica", "Tortura",
    "Sexual", "Tráfico", "Financeira",
    "Negligência", "Infantil", "Legal", "Outra"
]

COLS_REL = [
    "REL_PAI", "REL_MAE", "REL_PAD", "REL_CONJ", "REL_EXCON",
    "REL_NAMO", "REL_EXNAM", "REL_FILHO", "REL_DESCO",
    "REL_IRMAO", "REL_CONHEC", "REL_CUIDA", "REL_PATRAO",
    "REL_INST", "REL_POL", "REL_PROPRI", "REL_OUTROS"
]

print("✅ Constantes e diretórios configurados.")


# ============================================================
# SEÇÃO 1 — CARREGAMENTO E INSPEÇÃO INICIAL DOS DADOS
# ============================================================

## Seção 1 — Carregamento e Inspeção Inicial dos Dados

### Estratégia de ingestão
Os arquivos CSV do SINAN/DATASUS cobrem os anos de **2009 a 2025**
e são baixados diretamente do repositório *OpenScienceTechnology* no GitHub.
A leitura utiliza **encoding latin-1** (padrão do DATASUS) e filtragem
imediata pelo código de UF (`SG_UF_NOT == 50`) para reduzir o footprint
de memória, mantendo apenas os registros de **Mato Grosso do Sul**.

O campo `NU_IDADE_N` emprega codificação proprietária do SINAN:
- `4XXX` → anos de vida  
- `3XXX` → meses  
- `2XXX` → dias

In [ ]:
# ------------------------------------------------------------------
# 1.1  Função de download e leitura de um ano
# ------------------------------------------------------------------
def baixar_csv(ano: int, forcar: bool = False) -> Path:
    """
    Baixa o CSV de um ano do repositório GitHub e salva localmente.
    Retorna o Path do arquivo local.
    """
    url   = CSV_URLS[ano]
    dest  = DATA_DIR / f"VIOLBR{str(ano)[2:]}.csv"

    if dest.exists() and not forcar:
        logger.debug(f"Cache hit: {dest.name}")
        return dest

    logger.info(f"Baixando {dest.name} de {url}")
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()

    total = int(resp.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True,
        desc=f"VIOLBR{str(ano)[2:]}", leave=False
    ) as bar:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

    logger.success(f"Salvo: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


def ler_csv_ano(ano: int, apenas_ms: bool = True) -> pd.DataFrame:
    """
    Lê o CSV de um ano com as configurações corretas do SINAN/DATASUS.
    Se `apenas_ms=True`, filtra apenas registros com SG_UF_NOT == 50.
    """
    path = baixar_csv(ano)

    df = pd.read_csv(
        path,
        encoding="latin-1",
        low_memory=False,
        dtype=str,          # lê tudo como string para evitar perda de zeros
    )

    # Remove BOM UTF-8 da primeira coluna (artefato frequente do DATASUS)
    df.columns = [c.replace("\ufeff", "").replace("ï»¿", "").strip()
                  for c in df.columns]

    # Adiciona coluna de ano-fonte caso NU_ANO esteja ausente
    if "NU_ANO" not in df.columns:
        df["NU_ANO"] = str(ano)

    # Filtra MS
    if apenas_ms and "SG_UF_NOT" in df.columns:
        df = df[df["SG_UF_NOT"].astype(str).str.strip() == str(UF_MS)].copy()

    df["ANO_FONTE"] = ano
    logger.info(f"  {ano}: {len(df):>6,} registros MS")
    return df

In [ ]:
# ------------------------------------------------------------------
# 1.2  Carregamento consolidado 2009–2025
# ------------------------------------------------------------------
print("=" * 60)
print("📥 CARREGAMENTO DOS DADOS SINAN/DATASUS — 2009 a 2025")
print("=" * 60)

frames = []
for ano in ANOS:
    try:
        df_ano = ler_csv_ano(ano, apenas_ms=True)
        frames.append(df_ano)
    except Exception as e:
        logger.warning(f"Erro ao carregar {ano}: {e}")

df_raw = pd.concat(frames, ignore_index=True, sort=False)

print(f"\n📊 Dataset consolidado: {len(df_raw):,} registros | {df_raw.shape[1]} colunas")
print(f"   Período: {df_raw['ANO_FONTE'].astype(int).min()} – {df_raw['ANO_FONTE'].astype(int).max()}")
print(f"   Municípios distintos: {df_raw['ID_MUNICIP'].nunique()}")
logger.success(f"Dataset consolidado: {df_raw.shape}")

In [ ]:
# ------------------------------------------------------------------
# 1.3  Inspeção inicial
# ------------------------------------------------------------------
print("── head() ──")
display(df_raw.head(3))

print("\n── info() ──")
df_raw.info(verbose=False, show_counts=True)

print("\n── describe() (colunas numéricas após conversão prévia) ──")
cols_num_prev = ["NU_IDADE_N", "SG_UF_NOT", "SG_UF"]
df_prev = df_raw[cols_num_prev].apply(pd.to_numeric, errors="coerce")
display(df_prev.describe())

In [ ]:
# ------------------------------------------------------------------
# 1.4  Mapa de valores ausentes (missingno)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Seleciona colunas com missings parciais (entre 1% e 99%)
null_pct = df_raw.isnull().mean()
cols_parciais = null_pct[(null_pct > 0.01) & (null_pct < 0.99)].index.tolist()[:40]

msno.matrix(df_raw[cols_parciais].sample(min(500, len(df_raw)), random_state=42),
            ax=axes[0], sparkline=False, fontsize=7, color=(0.25, 0.45, 0.75))
axes[0].set_title("Matriz de Ausência — Colunas Relevantes", fontweight="bold")

msno.bar(df_raw[cols_parciais], ax=axes[1], fontsize=7,
         color="#4472C4", log=False)
axes[1].set_title("Completude por Coluna (%)", fontweight="bold")

plt.suptitle("SIPREV-Mulher/MS — Qualidade dos Dados Brutos (2009–2025)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_missings.png", bbox_inches="tight")
plt.show()

print(f"\nColunas 100% nulas: {(null_pct == 1).sum()}")
print(f"Colunas sem nenhum nulo: {(null_pct == 0).sum()}")
print(f"Colunas com missings parciais: {((null_pct > 0) & (null_pct < 1)).sum()}")


# ============================================================
# SEÇÃO 2 — LIMPEZA E PRÉ-PROCESSAMENTO
# ============================================================

## Seção 2 — Limpeza e Pré-processamento

O pipeline de tratamento aplica, em sequência:
1. **Conversão de tipos** — numérico e datetime
2. **Decodificação de idades** — campo `NU_IDADE_N` (SINAN)
3. **Padronização de categóricas** — raça/cor, local, escolaridade
4. **Harmonização municipal** — código IBGE 6 dígitos
5. **Criação de features derivadas** — faixa etária, flags de violência
6. **Validação de esquema** com Pandera

In [ ]:
# ------------------------------------------------------------------
# 2.1  Cópia de trabalho e conversão de tipos
# ------------------------------------------------------------------
df = df_raw.copy()

# Converte colunas numéricas-chave
NUM_COLS = [
    "SG_UF_NOT", "SG_UF", "CS_RACA", "CS_ESCOL_N", "CS_GESTANT",
    "NU_IDADE_N", "LOCAL_OCOR", "OUT_VEZES", "AUTOR_SEXO", "AUTOR_ALCO",
    "CLASSI_FIN", "EVOLUCAO", "NUM_ENVOLV",
] + COLS_VIOL + COLS_REL

for col in NUM_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Converte datas
DATE_COLS = ["DT_NOTIFIC", "DT_OCOR", "DT_OBITO", "DT_DIGITA", "DT_ENCERRA"]
for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

# ANO e MÊS a partir de DT_NOTIFIC (mais confiável que NU_ANO para séries)
df["ANO"]       = df["DT_NOTIFIC"].dt.year.fillna(df["ANO_FONTE"])
df["MES"]       = df["DT_NOTIFIC"].dt.month
df["TRIMESTRE"] = df["DT_NOTIFIC"].dt.quarter
df["PERIODO"]   = df["DT_NOTIFIC"].dt.to_period("M").astype(str)

logger.info("Conversão de tipos concluída.")

In [ ]:
# ------------------------------------------------------------------
# 2.2  Decodificação da idade (NU_IDADE_N)
# ------------------------------------------------------------------
def decodificar_idade(val):
    """
    Decodifica o campo NU_IDADE_N do SINAN:
      4XXX = anos  |  3XXX = meses  |  2XXX = dias  |  1XXX = horas
    Retorna a idade em anos (float).
    """
    if pd.isna(val):
        return np.nan
    v = int(val)
    if v >= 4000:
        return v - 4000
    elif v >= 3000:
        return (v - 3000) / 12
    elif v >= 2000:
        return (v - 2000) / 365
    elif v >= 1000:
        return 0.0
    return np.nan

df["IDADE_ANOS"] = df["NU_IDADE_N"].apply(decodificar_idade)

# Faixa etária
bins   = [-1, 11, 17, 29, 44, 59, 150]
labels = ["0–11", "12–17", "18–29", "30–44", "45–59", "60+"]
df["FAIXA_ETARIA"] = pd.cut(df["IDADE_ANOS"], bins=bins, labels=labels)

print("Distribuição de faixa etária em MS:")
print(df["FAIXA_ETARIA"].value_counts().sort_index())

In [ ]:
# ------------------------------------------------------------------
# 2.3  Decodificação de categóricas
# ------------------------------------------------------------------
# Raça/Cor
df["RACA_COR"]    = df["CS_RACA"].map(DICT_RACA).fillna("Ignorado")

# Escolaridade
df["ESCOLARIDADE"] = df["CS_ESCOL_N"].map(DICT_ESCOL).fillna("Ignorado")

# Local de ocorrência
df["LOCAL_DESC"]  = df["LOCAL_OCOR"].map(DICT_LOCAL).fillna("Outro")

# Sexo (já é string)
df["SEXO_DESC"]   = df["CS_SEXO"].map(DICT_SEXO).fillna("Ignorado")

# Evolução
df["EVOLUCAO_DESC"] = df["EVOLUCAO"].map(DICT_EVOLUCAO).fillna("Ignorado")

# Código do município: padroniza para 6 dígitos string
df["COD_MUN"]     = (
    df["ID_MUNICIP"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(6)
)

logger.info("Decodificação de categóricas concluída.")

In [ ]:
# ------------------------------------------------------------------
# 2.4  Flags de violência e variáveis derivadas
# ------------------------------------------------------------------
# Flags binárias limpas (1=Sim, 0=Não/Ignorado)
for col in COLS_VIOL:
    if col in df.columns:
        flag_col = col + "_FLAG"
        df[flag_col] = (df[col] == 1).astype(int)

# Contagem de tipos de violência por registro
flag_cols = [c + "_FLAG" for c in COLS_VIOL if c in df.columns]
df["N_TIPOS_VIOL"] = df[flag_cols].sum(axis=1)

# Flag: autor usou álcool (1=Sim)
df["ALCO_FLAG"] = (df["AUTOR_ALCO"] == 1).astype(int)

# Flag: reincidência
df["REINC_FLAG"] = (df["OUT_VEZES"] == 1).astype(int)

# Flag: vítima do sexo feminino
df["VITIMA_F"]   = (df["CS_SEXO"] == "F").astype(int)

# Flag: óbito por violência
df["OBITO_FLAG"] = (df["EVOLUCAO"] == 2).astype(int)

# Relação com agressor: agrupa em "íntimo", "familiar", "desconhecido"
df["REL_INTIMO"]  = (
    (df.get("REL_CONJ",  pd.Series(0)) == 1) |
    (df.get("REL_EXCON", pd.Series(0)) == 1) |
    (df.get("REL_NAMO",  pd.Series(0)) == 1) |
    (df.get("REL_EXNAM", pd.Series(0)) == 1)
).astype(int)

df["REL_FAMILIAR"] = (
    (df.get("REL_PAI",   pd.Series(0)) == 1) |
    (df.get("REL_MAE",   pd.Series(0)) == 1) |
    (df.get("REL_PAD",   pd.Series(0)) == 1) |
    (df.get("REL_FILHO", pd.Series(0)) == 1) |
    (df.get("REL_IRMAO", pd.Series(0)) == 1)
).astype(int)

df["REL_DESCO_FLAG"] = (df.get("REL_DESCO", pd.Series(np.nan)) == 1).astype(int)

print(f"Dataset após pré-processamento: {df.shape}")
logger.info(f"Pré-processamento finalizado: {df.shape}")


# ============================================================
# SIPREV-Mulher/MS — VERSÃO 2.0 (PATCH ROBUSTO)
# Correções e expansões sobre v1.0
# ============================================================
# MUDANÇAS PRINCIPAIS:
#  • Seção 2: flags de crimes sexuais específicos + feminicídio
#  • Seção 3 (expandida): tabelas com nomes de cidades,
#    comparativos por ano/mês, gráficos de crimes sexuais
#  • Seção 8.5: robusto com try/except e re-cálculo de SHAP
# ============================================================

# ============================================================
# PATCH 2 — Novas flags pós-pré-processamento (inserir após 2.4)
# ============================================================

## Seção 2 (adendo) — Flags de Crimes Sexuais e Feminicídio

O SINAN registra os subtipos de violência sexual nas colunas
`SEX_*` com codificação: **1 = Sim | 2 = Não | 8 = Não se aplica | 9 = Ignorado**

O **feminicídio** é inferido a partir de três critérios combinados:
1. Vítima do sexo feminino (`CS_SEXO == 'F'`)
2. Óbito por violência (`EVOLUCAO == 2`)
3. Agressor do convívio íntimo ou familiar

In [ ]:
# ------------------------------------------------------------------
# DICIONÁRIO COMPLETO DE MUNICÍPIOS DE MATO GROSSO DO SUL
# ------------------------------------------------------------------
MUNICIPIOS_MS = {
    "500025": "Água Clara",        "500055": "Alcinópolis",
    "500600": "Amambai",           "500700": "Anastácio",
    "500800": "Anaurilândia",      "500850": "Angélica",
    "500900": "Antônio João",      "501000": "Aparecida do Taboado",
    "501100": "Aquidauana",        "501150": "Aral Moreira",
    "501200": "Bandeirantes",      "501300": "Bataguassu",
    "501350": "Batayporã",         "501400": "Bela Vista",
    "501420": "Bodoquena",         "501500": "Bonito",
    "501600": "Brasilândia",       "501700": "Caarapó",
    "501800": "Camapuã",           "501900": "Campo Grande",
    "501950": "Caracol",           "502000": "Cassilândia",
    "502050": "Chapadão do Sul",   "502100": "Corguinho",
    "502150": "Coronel Sapucaia",  "502200": "Corumbá",
    "502300": "Costa Rica",        "502400": "Coxim",
    "502450": "Deodápolis",        "502460": "Dois Irmãos do Buriti",
    "502468": "Douradina",         "502500": "Dourados",
    "502536": "Eldorado",          "502560": "Fátima do Sul",
    "502572": "Figueirão",         "502600": "Glória de Dourados",
    "502700": "Guia Lopes da Laguna", "502800": "Iguatemi",
    "502850": "Inocência",         "502900": "Itaporã",
    "503000": "Itaquiraí",         "503100": "Ivinhema",
    "503110": "Japorã",            "503130": "Jaraguari",
    "503150": "Jardim",            "503200": "Jateí",
    "503250": "Juti",              "503300": "Ladário",
    "503350": "Laguna Carapã",     "503400": "Maracaju",
    "503500": "Miranda",           "503520": "Mundo Novo",
    "503536": "Naviraí",           "503544": "Nioaque",
    "503552": "Nova Alvorada do Sul", "503560": "Nova Andradina",
    "503600": "Novo Horizonte do Sul","503650": "Paraíso das Águas",
    "503700": "Paranaíba",         "503750": "Paranhos",
    "503800": "Pedro Gomes",       "503850": "Ponta Porã",
    "503900": "Porto Murtinho",    "503950": "Ribas do Rio Pardo",
    "504000": "Rio Brilhante",     "504100": "Rio Negro",
    "504200": "Rio Verde de Mato Grosso", "504300": "Rochedo",
    "504400": "Santa Rita do Pardo","504500": "São Gabriel do Oeste",
    "504600": "Selvíria",          "504700": "Sete Quedas",
    "504710": "Sidrolândia",       "504730": "Sonora",
    "504750": "Tacuru",            "504760": "Taquarussu",
    "504800": "Terenos",           "504900": "Três Lagoas",
    "505000": "Vicentina",
    # Campo Grande (código alternativo encontrado nos dados)
    "500270": "Campo Grande",
    # Outros códigos frequentes nos microdados
    "500320": "Dourados",          "500370": "Três Lagoas",
    "500830": "Ponta Porã",        "500620": "Corumbá",
    "500060": "Aquidauana",        "500660": "Naviraí",
    "500769": "Maracaju",          "500110": "Aparecida do Taboado",
    "500500": "Nova Andradina",    "500520": "Paranaíba",
    "500240": "Caarapó",           "500380": "Campo Grande (adj.)",
    "500570": "Sidrolândia",       "500790": "Chapadão do Sul",
    "500330": "Douradina",         "500630": "Camapuã",
    "500540": "Ribas do Rio Pardo","500090": "Amambai",
    "500470": "Itaquiraí",
}

# Adiciona nome do município ao DataFrame principal
df["NM_MUN"] = df["COD_MUN"].map(MUNICIPIOS_MS).fillna("Outro/MS")

print(f"Municípios identificados: {df['NM_MUN'].nunique()}")
print(f"Registros sem nome: {(df['NM_MUN'] == 'Outro/MS').sum():,}")

In [ ]:
# ------------------------------------------------------------------
# 2.A  Flags de crimes sexuais específicos (SINAN: 1=Sim, outros=Não)
# ------------------------------------------------------------------
# Colunas SEX_*: 1=Sim | 2=Não | 8=Não se aplica | 9=Ignorado
SEX_COLS_MAP = {
    "SEX_ASSEDI":  "Assédio Sexual",
    "SEX_ESTUPR":  "Estupro",
    "SEX_PUDOR":   "Atentado Violento ao Pudor",
    "SEX_PORNO":   "Pornografia / Exploração",
    "SEX_EXPLO":   "Exploração Sexual",
    "SEX_OUTRO":   "Violência Sexual (Outra)",
    "PEN_ORAL":    "Penetração Oral",
    "PEN_ANAL":    "Penetração Anal",
    "PEN_VAGINA":  "Penetração Vaginal",
}

for col, label in SEX_COLS_MAP.items():
    if col in df.columns:
        df[col + "_FLAG"] = (
            pd.to_numeric(df[col], errors="coerce") == 1
        ).astype(int)

# Flag composta: ESTUPRO (penetração ou tipo específico)
estupro_flags = ["SEX_ESTUPR_FLAG", "PEN_ORAL_FLAG", "PEN_ANAL_FLAG", "PEN_VAGINA_FLAG"]
estupro_flags = [c for c in estupro_flags if c in df.columns]
if estupro_flags:
    df["ESTUPRO_FLAG"] = df[estupro_flags].max(axis=1)

# Flag: Tentativa de estupro (violência sexual SEM penetração confirmada)
if "VIOL_SEXU_FLAG" in df.columns and "ESTUPRO_FLAG" in df.columns:
    df["TENT_ESTUPRO_FLAG"] = (
        (df["VIOL_SEXU_FLAG"] == 1) & (df["ESTUPRO_FLAG"] == 0)
    ).astype(int)

# Flag: Abuso sexual (assédio + pudor sem penetração)
abuso_flags = ["SEX_ASSEDI_FLAG", "SEX_PUDOR_FLAG"]
abuso_flags = [c for c in abuso_flags if c in df.columns]
if abuso_flags:
    df["ABUSO_SEXUAL_FLAG"] = df[abuso_flags].max(axis=1)

# Flag: Importunação sexual (assédio sem contato físico)
if "SEX_ASSEDI_FLAG" in df.columns:
    df["IMPORTUNACAO_SEXUAL_FLAG"] = df["SEX_ASSEDI_FLAG"]

# Flag: FEMINICÍDIO
# Critério primário: óbito por violência em mulher por agressor íntimo/familiar
df["FEMINICIDIO_FLAG"] = (
    (df["VITIMA_F"] == 1) &
    (df["OBITO_FLAG"] == 1) &
    (
        (df.get("REL_INTIMO",   pd.Series(0, index=df.index)) == 1) |
        (df.get("REL_FAMILIAR", pd.Series(0, index=df.index)) == 1)
    )
).astype(int)

# Flag: qualquer crime sexual (agregado)
crime_sex_flags = [c for c in df.columns
                   if c.endswith("_FLAG") and any(
                       x in c for x in ["SEX_","PEN_","ESTUPRO","ABUSO","IMPORTUNACAO"]
                   )]
if crime_sex_flags:
    df["CRIME_SEXUAL_TOTAL_FLAG"] = df[crime_sex_flags].max(axis=1)

# Sumário
print("=" * 60)
print("📊 FLAGS DE CRIMES SEXUAIS E FEMINICÍDIO — MS (2009–2025)")
print("=" * 60)
crimes_resumo = {
    "Violência Sexual (geral)":          df.get("VIOL_SEXU_FLAG",      pd.Series(0)).sum(),
    "Estupro (confirmado)":              df.get("ESTUPRO_FLAG",         pd.Series(0)).sum(),
    "Tentativa de Estupro":              df.get("TENT_ESTUPRO_FLAG",    pd.Series(0)).sum(),
    "Assédio Sexual":                    df.get("SEX_ASSEDI_FLAG",      pd.Series(0)).sum(),
    "Atentado Violento ao Pudor":        df.get("SEX_PUDOR_FLAG",       pd.Series(0)).sum(),
    "Abuso Sexual (assédio+pudor)":      df.get("ABUSO_SEXUAL_FLAG",    pd.Series(0)).sum(),
    "Importunação Sexual":               df.get("IMPORTUNACAO_SEXUAL_FLAG", pd.Series(0)).sum(),
    "Exploração Sexual":                 df.get("SEX_EXPLO_FLAG",       pd.Series(0)).sum(),
    "Feminicídio (óbito+mulher+agressor íntimo)": df.get("FEMINICIDIO_FLAG", pd.Series(0)).sum(),
}
for crime, total in crimes_resumo.items():
    pct = total / len(df) * 100
    print(f"  {crime:<45s} {total:>6,}  ({pct:.2f}%)")


# ============================================================
# SEÇÃO 3-EXTRA — ANÁLISES EXPANDIDAS COM NOMES DE CIDADES
# ============================================================

## Seção 3 (expandida) — Tabelas e Gráficos com Nomes das Cidades

Análises por município (com nome), por ano e por mês para todos os
tipos de crime, incluindo os crimes sexuais específicos identificados.

In [ ]:
# ------------------------------------------------------------------
# 3-E1  Tabela ranking — TODOS os 79 municípios com nome
# ------------------------------------------------------------------
COLS_CRIMES_EXT = {
    "VIOL_FISIC_FLAG":              "Violência Física",
    "VIOL_PSICO_FLAG":              "Violência Psicológica",
    "VIOL_SEXU_FLAG":               "Violência Sexual",
    "VIOL_NEGLI_FLAG":              "Negligência",
    "ESTUPRO_FLAG":                 "Estupro",
    "TENT_ESTUPRO_FLAG":            "Tentativa de Estupro",
    "ABUSO_SEXUAL_FLAG":            "Abuso Sexual",
    "IMPORTUNACAO_SEXUAL_FLAG":     "Importunação Sexual",
    "SEX_PUDOR_FLAG":               "At. Violento ao Pudor",
    "SEX_EXPLO_FLAG":               "Exploração Sexual",
    "FEMINICIDIO_FLAG":             "Feminicídio",
}
cols_ext_exist = {k: v for k, v in COLS_CRIMES_EXT.items() if k in df.columns}

agg_dict = {"N_TOTAL": ("COD_MUN", "count"),
            "N_FEMININO": ("VITIMA_F", "sum"),
            "REINC_TAXA": ("REINC_FLAG", "mean"),
            "ALCO_TAXA":  ("ALCO_FLAG",  "mean")}
for col in cols_ext_exist:
    agg_dict[col.replace("_FLAG","")] = (col, "sum")

tabela_cidades = (
    df.groupby("COD_MUN")
    .agg(**agg_dict)
    .reset_index()
)
tabela_cidades["NM_MUN"]  = tabela_cidades["COD_MUN"].map(MUNICIPIOS_MS).fillna("Outros/MS")
tabela_cidades["PCT_F"]   = (tabela_cidades["N_FEMININO"] / tabela_cidades["N_TOTAL"] * 100).round(1)
tabela_cidades = tabela_cidades.sort_values("N_TOTAL", ascending=False)

# Seleciona colunas para exibição
display_cols = (
    ["NM_MUN", "COD_MUN", "N_TOTAL", "N_FEMININO", "PCT_F", "REINC_TAXA", "ALCO_TAXA"]
    + [c.replace("_FLAG","") for c in cols_ext_exist]
)
display_cols = [c for c in display_cols if c in tabela_cidades.columns]

print("=" * 70)
print("📋 RANKING COMPLETO — 79 MUNICÍPIOS DE MS (2009–2025)")
print("=" * 70)
pd.set_option("display.max_rows", 85)
pd.set_option("display.max_columns", 25)
display(tabela_cidades[display_cols].head(79).style
    .background_gradient(subset=["N_TOTAL"], cmap="YlOrRd")
    .background_gradient(subset=["FEMINICIDIO"] if "FEMINICIDIO" in tabela_cidades.columns else [], cmap="Reds")
    .format({"PCT_F": "{:.1f}%", "REINC_TAXA": "{:.1%}", "ALCO_TAXA": "{:.1%}"})
)

# Salva em CSV
tabela_cidades[display_cols].to_csv(OUTPUT_DIR / "ranking_municipios_completo.csv",
                                     index=False, encoding="utf-8-sig")
print(f"\n✅ Tabela salva em outputs/ranking_municipios_completo.csv")

In [ ]:
# ------------------------------------------------------------------
# 3-E2  Crimes sexuais — prevalência detalhada (barras)
# ------------------------------------------------------------------
crimes_sex_totais = {v: df.get(k, pd.Series(0)).sum()
                     for k, v in cols_ext_exist.items()
                     if "SEX" in k or "ESTUPRO" in k or "ABUSO" in k
                     or "IMPORTUNACAO" in k or "FEMINICIDIO" in k
                     or "PUDOR" in k or "EXPLO" in k}

crimes_ser = pd.Series(crimes_sex_totais).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
paleta = plt.cm.RdPu(np.linspace(0.3, 0.95, len(crimes_ser)))
bars = crimes_ser.plot(kind="barh", ax=ax, color=paleta)
ax.set_xlabel("Número de Notificações (MS 2009–2025)", fontsize=11)
ax.set_title("Crimes Sexuais e Feminicídio — MS\n"
             "Classificação Detalhada por Tipo (SINAN/DATASUS)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, (val, name) in enumerate(zip(crimes_ser, crimes_ser.index)):
    ax.text(val + 5, i, f"{val:,}", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_crimes_sexuais.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E3  Evolução anual por tipo de crime sexual
# ------------------------------------------------------------------
crimes_sex_cols = [k for k in cols_ext_exist
                   if any(x in k for x in ["SEX_","ESTUPRO","ABUSO","IMPORTUNACAO",
                                            "FEMINICIDIO","PUDOR","EXPLO"])]

serie_crimes_ano = pd.DataFrame()
for col in crimes_sex_cols:
    if col in df.columns:
        s = df.groupby("ANO_FONTE")[col].sum().rename(cols_ext_exist.get(col, col))
        serie_crimes_ano = pd.concat([serie_crimes_ano, s], axis=1)

serie_crimes_ano = serie_crimes_ano.loc[
    serie_crimes_ano.index.astype(float).astype(int).isin(range(2009, 2026))
]

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Linhas por tipo
for col in serie_crimes_ano.columns:
    axes[0].plot(serie_crimes_ano.index.astype(str),
                 serie_crimes_ano[col], marker="o", linewidth=2, label=col)
axes[0].set_title("Evolução Anual — Crimes Sexuais e Feminicídio — MS",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Nº de Notificações")
axes[0].legend(fontsize=8, ncol=2)
axes[0].xaxis.set_tick_params(rotation=45)

# Área empilhada
serie_crimes_ano.plot(kind="area", stacked=True, ax=axes[1],
                       colormap="RdPu", alpha=0.8)
axes[1].set_title("Evolução Anual (Área Empilhada) — Crimes Sexuais",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Nº de Notificações")
axes[1].legend(fontsize=8, ncol=2, loc="upper left")
axes[1].xaxis.set_tick_params(rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_evolucao_anual_crimes_sexuais.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E4  Sazonalidade mensal — heatmap por crime × mês
# ------------------------------------------------------------------
df_sex_mes = df.dropna(subset=["MES"]).copy()
df_sex_mes["MES"] = df_sex_mes["MES"].astype(int)
MESES_PT = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
            7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}
df_sex_mes["MES_NM"] = df_sex_mes["MES"].map(MESES_PT)

heat_data = {}
for col in crimes_sex_cols:
    if col in df_sex_mes.columns:
        heat_data[cols_ext_exist.get(col, col)] = (
            df_sex_mes.groupby("MES")[col].sum()
        )

heat_df = pd.DataFrame(heat_data).T
heat_df.columns = [MESES_PT.get(c, c) for c in heat_df.columns]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(heat_df, annot=True, fmt=".0f", cmap="YlOrRd",
            ax=ax, linewidths=0.5, annot_kws={"size": 8})
ax.set_title("Sazonalidade Mensal — Crimes Sexuais e Feminicídio — MS (2009–2025)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Mês")
ax.set_ylabel("Tipo de Crime")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_sazonalidade_mensal.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E5  Top 20 cidades — cada tipo de crime sexual (tabela + gráfico)
# ------------------------------------------------------------------
for col, label in [
    ("ESTUPRO_FLAG", "Estupro"),
    ("FEMINICIDIO_FLAG", "Feminicídio"),
    ("ABUSO_SEXUAL_FLAG", "Abuso Sexual"),
]:
    if col not in df.columns:
        continue

    top20 = (
        df.groupby("COD_MUN")[col].sum()
        .reset_index()
        .assign(NM_MUN=lambda d: d["COD_MUN"].map(MUNICIPIOS_MS).fillna("Outro"))
        .sort_values(col, ascending=False)
        .head(20)
    )
    top20["LABEL"] = top20["NM_MUN"] + " (" + top20["COD_MUN"] + ")"

    fig, ax = plt.subplots(figsize=(12, 7))
    colors_t = plt.cm.Reds_r(np.linspace(0.2, 0.85, 20))
    bars = ax.barh(top20["LABEL"][::-1], top20[col][::-1], color=colors_t)
    ax.set_xlabel(f"Nº de Casos — {label}", fontsize=11)
    ax.set_title(f"Top 20 Municípios — {label}\nMS (2009–2025)", fontsize=12, fontweight="bold")
    for bar, val in zip(bars, top20[col][::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{int(val):,}", va="center", fontsize=9)
    plt.tight_layout()
    fname = f"03E_top20_{label.lower().replace(' ','_')}.png"
    plt.savefig(OUTPUT_DIR / fname, bbox_inches="tight")
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E6  Comparativo anual × mensal completo — tabela pivot
# ------------------------------------------------------------------
# Pivot: município × ano
pivot_mun_ano = (
    df.groupby(["NM_MUN", "ANO_FONTE"])
    .size()
    .unstack(fill_value=0)
)
pivot_mun_ano.columns = [int(c) for c in pivot_mun_ano.columns]
pivot_mun_ano = pivot_mun_ano.sort_values(
    pivot_mun_ano.columns.max(), ascending=False
)

print("=" * 70)
print("📅 NOTIFICAÇÕES POR MUNICÍPIO × ANO — MS (2009–2025)")
print("=" * 70)
display(pivot_mun_ano.style
    .background_gradient(cmap="Blues", axis=None)
    .format("{:,.0f}")
)
pivot_mun_ano.to_csv(OUTPUT_DIR / "pivot_municipio_ano.csv", encoding="utf-8-sig")

# Pivot: tipo de crime × ano
crimes_flag_cols = [c for c in df.columns if c.endswith("_FLAG")]
pivot_crime_ano = pd.DataFrame({
    cols_ext_exist.get(c, c.replace("_FLAG","")): df.groupby("ANO_FONTE")[c].sum()
    for c in crimes_flag_cols
    if c in cols_ext_exist
}).T
pivot_crime_ano.columns = [int(c) for c in pivot_crime_ano.columns]

print("\n\n📊 TIPOS DE CRIME × ANO — MS (2009–2025)")
display(pivot_crime_ano.style
    .background_gradient(cmap="YlOrRd", axis=None)
    .format("{:,.0f}")
)
pivot_crime_ano.to_csv(OUTPUT_DIR / "pivot_crime_ano.csv", encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------------
# 3-E7  Heatmap: municípios × meses (Campo Grande e top 15)
# ------------------------------------------------------------------
top15_mun = tabela_cidades.head(15)["NM_MUN"].tolist()
df_top15 = df[df["NM_MUN"].isin(top15_mun) & df["MES"].notna()]
df_top15["MES_INT"] = df_top15["MES"].astype(int)

pivot_mun_mes = (
    df_top15.groupby(["NM_MUN", "MES_INT"])
    .size()
    .unstack(fill_value=0)
)
pivot_mun_mes.columns = [MESES_PT.get(c, c) for c in pivot_mun_mes.columns]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_mun_mes, annot=True, fmt=",d", cmap="YlOrRd",
            ax=ax, linewidths=0.4, annot_kws={"size": 8})
ax.set_title("Notificações por Município × Mês\nTop 15 Municípios — MS (2009–2025)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Mês")
ax.set_ylabel("Município")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_heatmap_municipio_mes.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E8  Stacked bar — TOP 10 cidades × tipo de violência
# ------------------------------------------------------------------
top10_mun = tabela_cidades.head(10)["NM_MUN"].tolist()
df_top10 = df[df["NM_MUN"].isin(top10_mun)]

viol_top10 = pd.DataFrame()
for col, label in {
    "VIOL_FISIC_FLAG":          "Física",
    "VIOL_PSICO_FLAG":          "Psicológica",
    "ESTUPRO_FLAG":             "Estupro",
    "TENT_ESTUPRO_FLAG":        "Tent. Estupro",
    "ABUSO_SEXUAL_FLAG":        "Abuso Sexual",
    "IMPORTUNACAO_SEXUAL_FLAG": "Importunação",
    "SEX_PUDOR_FLAG":           "At. Pudor",
    "FEMINICIDIO_FLAG":         "Feminicídio",
    "VIOL_NEGLI_FLAG":          "Negligência",
}.items():
    if col in df_top10.columns:
        viol_top10[label] = df_top10.groupby("NM_MUN")[col].sum()

viol_top10 = viol_top10.loc[top10_mun]

fig, ax = plt.subplots(figsize=(16, 8))
viol_top10.plot(kind="bar", stacked=True, ax=ax, colormap="tab10", edgecolor="white")
ax.set_title("Composição dos Crimes por Tipo — Top 10 Municípios — MS",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Município")
ax.set_ylabel("Nº de Notificações")
ax.xaxis.set_tick_params(rotation=35)
ax.legend(title="Tipo de Crime", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_top10_composicao_crimes.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3-E9  Linha temporal: feminicídio + estupro por ano
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

for ax, col, label, color in [
    (axes[0], "FEMINICIDIO_FLAG", "Feminicídio", "#C0392B"),
    (axes[1], "ESTUPRO_FLAG",     "Estupro",     "#8E44AD"),
]:
    if col not in df.columns:
        continue
    serie = df.groupby("ANO_FONTE")[col].sum()
    serie = serie[serie.index.astype(float).astype(int).between(2009, 2025)]

    ax.bar(serie.index.astype(str), serie.values, color=color, alpha=0.75, edgecolor="white")
    ax.plot(serie.index.astype(str), serie.values, "o-", color="black", linewidth=1.5, markersize=5)
    ax.set_title(f"Evolução Anual — {label} — MS (2009–2025)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Nº de Casos")
    ax.xaxis.set_tick_params(rotation=45)
    for x, y in zip(range(len(serie)), serie.values):
        ax.text(x, y + 0.5, str(int(y)), ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03E_feminicidio_estupro_anual.png", bbox_inches="tight")
plt.show()


# ============================================================
# SEÇÃO 8.5 ROBUSTA — SHAP com fallback seguro
# ============================================================

In [ ]:
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# 2.5  Validação de esquema com Pandera
# ------------------------------------------------------------------
schema = pda.DataFrameSchema(
    columns={
        "COD_MUN":      pda.Column(str, pda.Check.str_length(6, 7)),
        "ANO":          pda.Column(float, pda.Check.in_range(2008, 2026),
                                   nullable=True),
        "CS_SEXO":      pda.Column(str,
                                   pda.Check.isin(["F", "M", "I"]),
                                   nullable=True),
        "VITIMA_F":     pda.Column(int, pda.Check.isin([0, 1])),
        "REINC_FLAG":   pda.Column(int, pda.Check.isin([0, 1])),
        "ALCO_FLAG":    pda.Column(int, pda.Check.isin([0, 1])),
        "N_TIPOS_VIOL": pda.Column(int, pda.Check.greater_than_or_equal_to(0)),
    },
    coerce=True,
    strict=False,       # permite colunas extras
)

try:
    schema.validate(df, lazy=True)
    print("✅ Validação Pandera: esquema aprovado sem erros.")
except pda.errors.SchemaErrors as errs:
    print(f"⚠️  Falhas de validação: {len(errs.failure_cases)}")
    display(errs.failure_cases.head(10))

In [ ]:
# ------------------------------------------------------------------
# 2.6  Persistência em Parquet particionado por ano
# ------------------------------------------------------------------
table = pa.Table.from_pandas(df)
pq.write_to_dataset(
    table,
    root_path=str(PROC_DIR / "siprev_ms"),
    partition_cols=["ANO_FONTE"],
    compression="snappy",
)
# Também salva consolidado
df.to_parquet(PROC_DIR / "siprev_ms_consolidado.parquet",
              index=False, compression="snappy")

print(f"✅ Dados processados salvos em {PROC_DIR}")
logger.success("Dados processados persistidos em Parquet.")


# ============================================================
# SEÇÃO 3 — ANÁLISE EXPLORATÓRIA DESCRITIVA (AED)
# ============================================================

## Seção 3 — Análise Exploratória Descritiva

Responde sistematicamente às **7 questões científicas** exigidas,
combinando estatísticas descritivas, testes de hipótese e visualizações.

In [ ]:
# ------------------------------------------------------------------
# 3.1  Ranking de municípios por notificações
# ------------------------------------------------------------------
rank_mun = (
    df.groupby("COD_MUN")
    .agg(N_NOTIF=("COD_MUN", "count"),
         N_FEMININO=("VITIMA_F", "sum"),
         REINC_MEDIA=("REINC_FLAG", "mean"),
         ALCO_MEDIA=("ALCO_FLAG", "mean"))
    .reset_index()
    .sort_values("N_NOTIF", ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 6))
top15 = rank_mun.head(15)
bars = ax.barh(top15["COD_MUN"][::-1], top15["N_NOTIF"][::-1],
               color=plt.cm.Blues_r(np.linspace(0.3, 0.9, 15)))
ax.set_xlabel("Número de Notificações (2009–2025)", fontsize=11)
ax.set_title("Top 15 Municípios de MS por Volume de Notificações\n"
             "Violência Interpessoal/Autoprovocada — SINAN", fontsize=12, fontweight="bold")
for bar, val in zip(bars, top15["N_NOTIF"][::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_ranking_municipios.png", bbox_inches="tight")
plt.show()

display(rank_mun.head(20))

In [ ]:
# ------------------------------------------------------------------
# 3.2  Prevalência por tipo de violência
# ------------------------------------------------------------------
viol_totais = pd.Series(
    {label: df[col + "_FLAG"].sum()
     for col, label in zip(COLS_VIOL, LABELS_VIOL)
     if (col + "_FLAG") in df.columns},
    name="Total"
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#C0392B" if v == viol_totais.max() else "#2E86AB"
          for v in viol_totais]
viol_totais.plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Número de Notificações", fontsize=11)
ax.set_title("Prevalência por Tipo de Violência — MS (2009–2025)", fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, (val, name) in enumerate(zip(viol_totais, viol_totais.index)):
    pct = val / viol_totais.sum() * 100
    ax.text(val + 20, i, f"{pct:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_prevalencia_violencia.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.3  Série temporal — tendência anual e mensal
# ------------------------------------------------------------------
serie_anual = df.groupby("ANO_FONTE").size().reset_index(name="N_NOTIF")
serie_anual = serie_anual[serie_anual["ANO_FONTE"].between(2009, 2025)]

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

# Anual
axes[0].plot(serie_anual["ANO_FONTE"], serie_anual["N_NOTIF"],
             marker="o", color="#2E86AB", linewidth=2.5)
axes[0].fill_between(serie_anual["ANO_FONTE"], serie_anual["N_NOTIF"],
                     alpha=0.15, color="#2E86AB")
axes[0].set_title("Notificações Anuais — MS (2009–2025)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Nº de Notificações")
axes[0].set_xticks(serie_anual["ANO_FONTE"])
axes[0].xaxis.set_tick_params(rotation=45)
# Linha de tendência
z = np.polyfit(serie_anual["ANO_FONTE"], serie_anual["N_NOTIF"], 1)
p = np.poly1d(z)
axes[0].plot(serie_anual["ANO_FONTE"], p(serie_anual["ANO_FONTE"]),
             "--", color="red", alpha=0.6, label="Tendência linear")
axes[0].legend()

# Mensal (série completa)
df_notif = df.dropna(subset=["DT_NOTIFIC"]).copy()
serie_mensal = df_notif.set_index("DT_NOTIFIC").resample("M").size()
serie_mensal.plot(ax=axes[1], color="#E74C3C", linewidth=1.5)
axes[1].set_title("Notificações Mensais — MS (2009–2025)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Nº de Notificações")
axes[1].set_xlabel("Mês/Ano")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_serie_temporal.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.4  Decomposição de série temporal (tendência, sazonalidade, resíduos)
# ------------------------------------------------------------------
serie_m = serie_mensal.copy()
serie_m = serie_m[serie_m > 0].asfreq("M").fillna(method="ffill")

if len(serie_m) >= 24:
    decomp = seasonal_decompose(serie_m, model="additive", period=12)
    fig = decomp.plot()
    fig.set_size_inches(14, 10)
    fig.suptitle("Decomposição da Série Temporal — Notificações MS",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_decomposicao_serie.png", bbox_inches="tight")
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.5  Distribuição etária — histograma e tabela cruzada
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histograma geral
df["IDADE_ANOS"].dropna().clip(0, 80).plot(
    kind="hist", bins=40, ax=axes[0], color="#8E44AD", edgecolor="white")
axes[0].set_title("Distribuição de Idade das Vítimas — MS", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Idade (anos)")
axes[0].set_ylabel("Frequência")
axes[0].axvline(df["IDADE_ANOS"].median(), color="red", linestyle="--",
                label=f"Mediana: {df['IDADE_ANOS'].median():.0f} anos")
axes[0].legend()

# Faixa etária × tipo de violência (top 4)
top_viol = ["VIOL_FISIC_FLAG", "VIOL_PSICO_FLAG", "VIOL_SEXU_FLAG", "VIOL_NEGLI_FLAG"]
top_labels = ["Física", "Psicológica", "Sexual", "Negligência"]
tab_faixa = pd.DataFrame({
    label: df.groupby("FAIXA_ETARIA")[col].mean()
    for col, label in zip(top_viol, top_labels)
    if col in df.columns
})
tab_faixa.plot(kind="bar", ax=axes[1], colormap="Set2", edgecolor="white")
axes[1].set_title("Taxa por Faixa Etária × Tipo de Violência", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Faixa Etária")
axes[1].set_ylabel("Proporção de Casos")
axes[1].legend(title="Tipo")
axes[1].xaxis.set_tick_params(rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_faixas_etarias.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.6  Local de ocorrência
# ------------------------------------------------------------------
local_dist = df["LOCAL_DESC"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
colors_l = ["#E74C3C" if v == local_dist.max() else "#AED6F1"
            for v in local_dist]
local_dist.plot(kind="bar", ax=ax, color=colors_l, edgecolor="white")
ax.set_title("Local de Ocorrência da Violência — MS (2009–2025)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Nº de Notificações")
ax.xaxis.set_tick_params(rotation=30)
for i, v in enumerate(local_dist):
    ax.text(i, v + 20, f"{v/len(df)*100:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_local_ocorrencia.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 3.7  Reincidência × álcool × gravidade — heatmap de correlações Spearman
# ------------------------------------------------------------------
corr_cols = ["REINC_FLAG", "ALCO_FLAG", "N_TIPOS_VIOL", "OBITO_FLAG",
             "VIOL_FISIC_FLAG", "VIOL_PSICO_FLAG", "VIOL_SEXU_FLAG",
             "REL_INTIMO", "REL_FAMILIAR", "VITIMA_F"]
corr_cols = [c for c in corr_cols if c in df.columns]

corr_mat = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(
    corr_mat, mask=mask, ax=ax,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    linewidths=0.5,
    xticklabels=[c.replace("_FLAG","") for c in corr_cols],
    yticklabels=[c.replace("_FLAG","") for c in corr_cols],
)
ax.set_title("Correlações de Spearman — Variáveis Epidemiológicas — MS",
             fontsize=12, fontweight="bold")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_correlacoes_spearman.png", bbox_inches="tight")
plt.show()

# Testes qui-quadrado: Reincidência × Uso de álcool
if "REINC_FLAG" in df.columns and "ALCO_FLAG" in df.columns:
    tab_cont = pd.crosstab(df["REINC_FLAG"], df["ALCO_FLAG"])
    chi2, p_val, dof, _ = stats.chi2_contingency(tab_cont)
    print(f"\nTeste Qui-Quadrado — Reincidência × Álcool:")
    print(f"  χ² = {chi2:.2f} | p-valor = {p_val:.4e} | gl = {dof}")
    print(f"  {'✅ Associação estatisticamente significativa (p<0.05)' if p_val < 0.05 else '❌ Sem associação significativa'}")

In [ ]:
# ------------------------------------------------------------------
# 3.8  Perfil racial e educacional
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raça/Cor
raca_dist = df["RACA_COR"].value_counts()
axes[0].pie(raca_dist, labels=raca_dist.index,
            autopct="%1.1f%%", startangle=90,
            colors=sns.color_palette("Set3", len(raca_dist)))
axes[0].set_title("Raça/Cor das Vítimas — MS", fontsize=11, fontweight="bold")

# Escolaridade
escol_dist = df["ESCOLARIDADE"].value_counts()
escol_dist[escol_dist > 50].sort_values().plot(
    kind="barh", ax=axes[1], color="#27AE60", edgecolor="white")
axes[1].set_title("Escolaridade das Vítimas — MS", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Nº de Notificações")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_perfil_racial_educacional.png", bbox_inches="tight")
plt.show()

print("\n✅ Seção 3 (AED) concluída — gráficos salvos em outputs/")


# ============================================================
# SEÇÃO 4 — ANÁLISE GEOESPACIAL
# ============================================================

## Seção 4 — Análise Geoespacial

Utiliza **GeoPandas** + **Folium** para visualizar a distribuição
territorial das notificações nos **79 municípios de MS**.
O shapefile municipal é obtido diretamente do IBGE.

In [ ]:
# ------------------------------------------------------------------
# 4.1  Download do shapefile de municípios do MS (IBGE)
# ------------------------------------------------------------------
SHP_URL = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
    "malhas_municipais/municipio_2022/UFs/MS/"
    "MS_Municipios_2022.zip"
)
shp_dest = DATA_DIR / "MS_Municipios.zip"

if not shp_dest.exists():
    print("Baixando shapefile de municípios do MS (IBGE)...")
    r = requests.get(SHP_URL, stream=True, timeout=120)
    with open(shp_dest, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    print("✅ Shapefile baixado.")

import zipfile
shp_dir = DATA_DIR / "shapefile_ms"
shp_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(shp_dest) as z:
    z.extractall(shp_dir)

shp_file = next(shp_dir.glob("*.shp"))
gdf_ms = gpd.read_file(shp_file, encoding="utf-8")
gdf_ms = gdf_ms.to_crs(epsg=4326)   # WGS-84

# Padroniza código IBGE para 6 dígitos (SINAN usa 6, IBGE geralmente 7)
gdf_ms["CD_MUN6"] = gdf_ms["CD_MUN"].astype(str).str[:6]

print(f"Shapefile: {len(gdf_ms)} municípios | CRS: {gdf_ms.crs}")
display(gdf_ms[["CD_MUN", "NM_MUN", "geometry"]].head())

In [ ]:
# ------------------------------------------------------------------
# 4.2  Join entre shapefile e dados SINAN
# ------------------------------------------------------------------
notif_mun = (
    df.groupby("COD_MUN")
    .agg(
        N_NOTIF          = ("COD_MUN",        "count"),
        VIOL_FISIC_PROP  = ("VIOL_FISIC_FLAG","mean"),
        VIOL_PSICO_PROP  = ("VIOL_PSICO_FLAG","mean"),
        VIOL_SEXU_PROP   = ("VIOL_SEXU_FLAG", "mean"),
        REINC_TAXA       = ("REINC_FLAG",      "mean"),
        ALCO_TAXA        = ("ALCO_FLAG",       "mean"),
        OBITO_TAXA       = ("OBITO_FLAG",      "mean"),
        VITIMA_F_PROP    = ("VITIMA_F",        "mean"),
        IDADE_MEDIA      = ("IDADE_ANOS",      "mean"),
    )
    .reset_index()
)
notif_mun["COD_MUN"] = notif_mun["COD_MUN"].astype(str).str.zfill(6)

gdf_dados = gdf_ms.merge(notif_mun, left_on="CD_MUN6", right_on="COD_MUN", how="left")
gdf_dados["N_NOTIF"] = gdf_dados["N_NOTIF"].fillna(0)

print(f"Municípios com dados: {(gdf_dados['N_NOTIF'] > 0).sum()} de {len(gdf_dados)}")

In [ ]:
# ------------------------------------------------------------------
# 4.3  Mapa coroplético — total de notificações (Matplotlib)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, col, title, cmap in [
    (axes[0], "N_NOTIF",    "Total de Notificações",    "YlOrRd"),
    (axes[1], "REINC_TAXA", "Taxa de Reincidência (%)", "PuRd"),
]:
    gdf_dados.plot(column=col, ax=ax, cmap=cmap, legend=True,
                   edgecolor="gray", linewidth=0.3, missing_kwds={"color": "#DDDDDD"})
    ax.set_title(f"MS — {title}\n(2009–2025)", fontsize=12, fontweight="bold")
    ax.set_axis_off()
    try:
        ctx.add_basemap(ax, crs=gdf_dados.crs.to_string(),
                        source=ctx.providers.CartoDB.Positron, zoom=7)
    except Exception:
        pass

plt.suptitle("SIPREV-Mulher/MS — Distribuição Geoespacial",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_mapa_coropletico.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 4.4  Mapa interativo Folium
# ------------------------------------------------------------------
m = folium.Map(
    location=[-20.5, -54.6],
    zoom_start=7,
    tiles="CartoDB positron",
)

# Coroplético de notificações
folium.Choropleth(
    geo_data    = gdf_dados.__geo_interface__,
    name        = "Total de Notificações",
    data        = gdf_dados[["CD_MUN6", "N_NOTIF"]],
    columns     = ["CD_MUN6", "N_NOTIF"],
    key_on      = "feature.properties.CD_MUN6",
    fill_color  = "YlOrRd",
    fill_opacity= 0.75,
    line_opacity= 0.3,
    legend_name = "Nº de Notificações (2009–2025)",
    nan_fill_color = "#AAAAAA",
).add_to(m)

# Tooltip com nome do município
folium.GeoJson(
    gdf_dados,
    name="Municípios",
    tooltip=folium.GeoJsonTooltip(
        fields=["NM_MUN", "N_NOTIF", "REINC_TAXA", "OBITO_TAXA"],
        aliases=["Município", "Notificações", "Taxa Reincidência", "Taxa Óbito"],
        localize=True,
    ),
    style_function=lambda x: {"fillOpacity": 0, "weight": 0.5, "color": "#555"},
).add_to(m)

# HeatMap de intensidade
coords_heat = []
df_coords = df.dropna(subset=["DT_NOTIFIC"]).copy()
# Usa centroide do município como proxy de coordenada
centroids = gdf_dados[["CD_MUN6","geometry"]].copy()
centroids["lat"] = centroids.geometry.centroid.y
centroids["lon"] = centroids.geometry.centroid.x
coord_map = centroids.set_index("CD_MUN6")[["lat","lon"]].to_dict("index")

for _, row in df_coords.iterrows():
    cod = str(row.get("COD_MUN","")).zfill(6)
    if cod in coord_map:
        coords_heat.append([coord_map[cod]["lat"], coord_map[cod]["lon"], 1])

if coords_heat:
    HeatMap(coords_heat[:5000], name="Heatmap Intensidade",
            radius=12, blur=10, max_zoom=13).add_to(m)

folium.LayerControl().add_to(m)

map_path = OUTPUT_DIR / "04_mapa_interativo.html"
m.save(str(map_path))
print(f"✅ Mapa interativo salvo em: {map_path}")
print("   Abra o arquivo HTML no navegador para interagir.")


# ============================================================
# SEÇÃO 5 — CLUSTERIZAÇÃO MUNICIPAL
# ============================================================

## Seção 5 — Clusterização Municipal

Agrupa os **79 municípios** de MS por perfil epidemiológico de violência
usando **K-Means**, **DBSCAN** e **HDBSCAN**.

In [ ]:
# ------------------------------------------------------------------
# 5.1  Feature matrix por município
# ------------------------------------------------------------------
feat_mun = (
    df.groupby("COD_MUN")
    .agg(
        N_NOTIF          = ("COD_MUN",         "count"),
        VITIMA_F_PROP    = ("VITIMA_F",         "mean"),
        VIOL_FISIC_PROP  = ("VIOL_FISIC_FLAG",  "mean"),
        VIOL_PSICO_PROP  = ("VIOL_PSICO_FLAG",  "mean"),
        VIOL_SEXU_PROP   = ("VIOL_SEXU_FLAG",   "mean"),
        VIOL_NEGLI_PROP  = ("VIOL_NEGLI_FLAG",  "mean"),
        REINC_TAXA       = ("REINC_FLAG",        "mean"),
        ALCO_TAXA        = ("ALCO_FLAG",         "mean"),
        OBITO_TAXA       = ("OBITO_FLAG",        "mean"),
        REL_INTIMO_PROP  = ("REL_INTIMO",        "mean"),
        REL_FAM_PROP     = ("REL_FAMILIAR",      "mean"),
        IDADE_MEDIA      = ("IDADE_ANOS",        "mean"),
        N_TIPOS_MEDIA    = ("N_TIPOS_VIOL",      "mean"),
    )
    .reset_index()
    .fillna(0)
)

feat_cols = [c for c in feat_mun.columns if c != "COD_MUN"]
X_mun = feat_mun[feat_cols].values

scaler_mun = StandardScaler()
X_scaled   = scaler_mun.fit_transform(X_mun)

print(f"Feature matrix para clusterização: {X_scaled.shape}")
display(feat_mun.describe())

In [ ]:
# ------------------------------------------------------------------
# 5.2  K-Means — Elbow + Silhouette
# ------------------------------------------------------------------
inertias, silhouettes = [], []
K_RANGE = range(2, 12)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_RANGE), inertias, marker="o", color="#2E86AB")
axes[0].set_title("Método do Cotovelo (Elbow)", fontweight="bold")
axes[0].set_xlabel("Número de Clusters (k)")
axes[0].set_ylabel("Inércia")

axes[1].plot(list(K_RANGE), silhouettes, marker="s", color="#E74C3C")
axes[1].set_title("Silhouette Score por k", fontweight="bold")
axes[1].set_xlabel("Número de Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].axvline(np.argmax(silhouettes) + 2, color="gray", linestyle="--",
                label=f"Melhor k={np.argmax(silhouettes)+2}")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_elbow_silhouette.png", bbox_inches="tight")
plt.show()

K_OTM = int(np.argmax(silhouettes)) + 2
print(f"\n✅ k ótimo selecionado: {K_OTM} (Silhouette = {max(silhouettes):.3f})")

In [ ]:
# ------------------------------------------------------------------
# 5.3  Ajuste final do K-Means
# ------------------------------------------------------------------
km_final = KMeans(n_clusters=K_OTM, random_state=42, n_init=10)
feat_mun["CLUSTER_KMEANS"] = km_final.fit_predict(X_scaled)

# DBSCAN
dbscan = DBSCAN(eps=1.2, min_samples=3)
feat_mun["CLUSTER_DBSCAN"] = dbscan.fit_predict(X_scaled)

# HDBSCAN
hdb = hdbscan.HDBSCAN(min_cluster_size=4, min_samples=2,
                      cluster_selection_method="eom")
feat_mun["CLUSTER_HDBSCAN"] = hdb.fit_predict(X_scaled)

print("Distribuição K-Means:")
print(feat_mun["CLUSTER_KMEANS"].value_counts().sort_index())
print("\nDistribuição DBSCAN (-1 = ruído):")
print(feat_mun["CLUSTER_DBSCAN"].value_counts().sort_index())
print("\nDistribuição HDBSCAN (-1 = ruído):")
print(feat_mun["CLUSTER_HDBSCAN"].value_counts().sort_index())

In [ ]:
# ------------------------------------------------------------------
# 5.4  Mapa de clusters — K-Means
# ------------------------------------------------------------------
CLUSTER_NOMES = {
    0: "Baixa notificação / Baixa reincidência",
    1: "Alta violência física e psicológica",
    2: "Alta incidência de violência sexual",
    3: "Alta reincidência / Agressor do convívio",
    4: "Perfil misto / Em transição",
    5: "Negligência predominante",
}

feat_mun["CLUSTER_NOME"] = feat_mun["CLUSTER_KMEANS"].map(
    lambda x: CLUSTER_NOMES.get(x, f"Cluster {x}")
)

gdf_clusters = gdf_ms.merge(
    feat_mun[["COD_MUN", "CLUSTER_KMEANS", "CLUSTER_NOME", "N_NOTIF"]],
    left_on="CD_MUN6", right_on="COD_MUN", how="left"
)

fig, ax = plt.subplots(figsize=(12, 10))
gdf_clusters.plot(
    column="CLUSTER_KMEANS", ax=ax,
    cmap="Set1", legend=True,
    legend_kwds={"title": "Cluster K-Means",
                 "bbox_to_anchor": (1.3, 1)},
    edgecolor="white", linewidth=0.4,
    missing_kwds={"color": "#CCCCCC", "label": "Sem dados"},
)
ax.set_title("Clusterização Municipal — K-Means\nMS (2009–2025)",
             fontsize=13, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_mapa_clusters.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 5.5  Radar chart por cluster
# ------------------------------------------------------------------
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_features = [
    "VIOL_FISIC_PROP","VIOL_PSICO_PROP","VIOL_SEXU_PROP",
    "REINC_TAXA","ALCO_TAXA","OBITO_TAXA","REL_INTIMO_PROP",
]
cluster_means = feat_mun.groupby("CLUSTER_KMEANS")[radar_features].mean()

N_feat = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N_feat, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
colors_r = plt.cm.Set1(np.linspace(0, 1, len(cluster_means)))

for i, (idx, row) in enumerate(cluster_means.iterrows()):
    vals = row.tolist() + [row.iloc[0]]
    ax.plot(angles, vals, color=colors_r[i], linewidth=2, label=CLUSTER_NOMES.get(idx, f"C{idx}"))
    ax.fill(angles, vals, color=colors_r[i], alpha=0.15)

ax.set_xticks(angles[:-1])
labels_r = ["Viol. Física","Viol. Psicol.","Viol. Sexual",
            "Reincidência","Álcool","Óbito","Rel. Íntima"]
ax.set_xticklabels(labels_r, size=9)
ax.set_title("Perfil dos Clusters — K-Means", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_radar_clusters.png", bbox_inches="tight")
plt.show()

print("\n✅ Seção 5 (Clusterização) concluída.")



# ============================================================
# SEÇÃO 6 — MODELOS PREDITIVOS CLÁSSICOS (ML)
# ============================================================

## Seção 6 — Modelos Preditivos Clássicos

Implementa quatro frentes de modelagem supervisionada:
1. **Regressão** — previsão do volume de casos mensais
2. **Classificação** — risco municipal (Baixo / Médio / Alto)
3. **Detecção de anomalias** — IsolationForest / LOF
4. **Séries temporais** — Prophet + ARIMA/SARIMA

In [ ]:
# ------------------------------------------------------------------
# 6.1  Construção do dataset panel (município × mês)
# ------------------------------------------------------------------
df_panel = (
    df.dropna(subset=["DT_NOTIFIC","COD_MUN"])
    .assign(
        ANO_MES=lambda d: d["DT_NOTIFIC"].dt.to_period("M")
    )
    .groupby(["COD_MUN","ANO_MES"])
    .agg(
        N_NOTIF         = ("COD_MUN",        "count"),
        VIOL_FISIC_PROP = ("VIOL_FISIC_FLAG","mean"),
        VIOL_PSICO_PROP = ("VIOL_PSICO_FLAG","mean"),
        VIOL_SEXU_PROP  = ("VIOL_SEXU_FLAG", "mean"),
        REINC_TAXA      = ("REINC_FLAG",      "mean"),
        ALCO_TAXA       = ("ALCO_FLAG",       "mean"),
        VITIMA_F_PROP   = ("VITIMA_F",        "mean"),
        IDADE_MEDIA     = ("IDADE_ANOS",      "mean"),
    )
    .reset_index()
)

df_panel["ANO"]       = df_panel["ANO_MES"].dt.year
df_panel["MES"]       = df_panel["ANO_MES"].dt.month
df_panel["TRIMESTRE"] = df_panel["MES"].apply(lambda m: (m-1)//3 + 1)
df_panel["TENDENCIA"] = (df_panel["ANO"] - 2009) * 12 + df_panel["MES"]

# Lag features (1 e 3 meses)
df_panel = df_panel.sort_values(["COD_MUN","ANO_MES"])
df_panel["N_LAG1"]    = df_panel.groupby("COD_MUN")["N_NOTIF"].shift(1)
df_panel["N_LAG3"]    = df_panel.groupby("COD_MUN")["N_NOTIF"].shift(3)
df_panel["N_MEDIA3M"] = df_panel.groupby("COD_MUN")["N_NOTIF"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

df_panel = df_panel.dropna(subset=["N_LAG1"])

# Rótulo de risco para classificação
df_panel["RISCO_LABEL"] = pd.qcut(
    df_panel["N_NOTIF"], q=3,
    labels=["Baixo", "Médio", "Alto"]
)
RISCO_MAP = {"Baixo": 0, "Médio": 1, "Alto": 2}
df_panel["RISCO_NUM"] = df_panel["RISCO_LABEL"].map(RISCO_MAP)

print(f"Dataset panel: {df_panel.shape}")
print(df_panel["RISCO_LABEL"].value_counts())

In [ ]:
# ------------------------------------------------------------------
# 6.2  Preparação de features e split temporal
# ------------------------------------------------------------------
FEAT_COLS = [
    "ANO","MES","TRIMESTRE","TENDENCIA",
    "N_LAG1","N_LAG3","N_MEDIA3M",
    "VIOL_FISIC_PROP","VIOL_PSICO_PROP","VIOL_SEXU_PROP",
    "REINC_TAXA","ALCO_TAXA","VITIMA_F_PROP","IDADE_MEDIA",
]
FEAT_COLS = [c for c in FEAT_COLS if c in df_panel.columns]

df_ml = df_panel.dropna(subset=FEAT_COLS + ["N_NOTIF","RISCO_NUM"])

# Split temporal: últimos 20% para teste
split_idx = int(len(df_ml) * 0.80)
df_train  = df_ml.iloc[:split_idx]
df_test   = df_ml.iloc[split_idx:]

X_train   = df_train[FEAT_COLS].values
X_test    = df_test[FEAT_COLS].values
y_reg_tr  = df_train["N_NOTIF"].values
y_reg_te  = df_test["N_NOTIF"].values
y_cls_tr  = df_train["RISCO_NUM"].values.astype(int)
y_cls_te  = df_test["RISCO_NUM"].values.astype(int)

scaler_ml = StandardScaler()
X_train_s = scaler_ml.fit_transform(X_train)
X_test_s  = scaler_ml.transform(X_test)

# SMOTE para classificação (desbalanceamento)
try:
    sm = SMOTE(random_state=42, k_neighbors=min(3, min(np.bincount(y_cls_tr))-1))
    X_tr_sm, y_tr_sm = sm.fit_resample(X_train_s, y_cls_tr)
    print(f"SMOTE: {len(y_cls_tr)} → {len(y_tr_sm)} amostras")
except Exception as e:
    X_tr_sm, y_tr_sm = X_train_s, y_cls_tr
    print(f"SMOTE não aplicado: {e}")

print(f"\nTreino: {len(X_train)} | Teste: {len(X_test)}")

In [ ]:
# ------------------------------------------------------------------
# 6.3  Treinamento — Regressão (previsão de casos)
# ------------------------------------------------------------------
mlflow.set_experiment("SIPREV_Regressao")

REG_MODELS = {
    "RandomForest":  RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":       xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                                       random_state=42, verbosity=0),
    "LightGBM":      lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                        random_state=42, verbose=-1),
}

resultados_reg = {}

for nome, modelo in REG_MODELS.items():
    with mlflow.start_run(run_name=f"REG_{nome}"):
        tscv = TimeSeriesSplit(n_splits=5)
        cv_scores = cross_val_score(modelo, X_train_s, y_reg_tr,
                                    cv=tscv, scoring="neg_mean_absolute_error")

        modelo.fit(X_train_s, y_reg_tr)
        y_pred = modelo.predict(X_test_s)

        mae  = mean_absolute_error(y_reg_te, y_pred)
        rmse = np.sqrt(mean_squared_error(y_reg_te, y_pred))
        r2   = r2_score(y_reg_te, y_pred)

        mlflow.log_params({"model": nome})
        mlflow.log_metrics({"MAE": mae, "RMSE": rmse, "R2": r2})
        mlflow.sklearn.log_model(modelo, f"model_{nome}")

        resultados_reg[nome] = {"MAE": mae, "RMSE": rmse, "R²": r2}
        print(f"  {nome:15s} | MAE={mae:.2f} | RMSE={rmse:.2f} | R²={r2:.3f}")

# Salva melhor modelo
melhor_reg = min(resultados_reg, key=lambda k: resultados_reg[k]["MAE"])
joblib.dump(REG_MODELS[melhor_reg], MODEL_DIR / f"reg_{melhor_reg.lower()}.joblib")

df_res_reg = pd.DataFrame(resultados_reg).T
print(f"\n🏆 Melhor regressão: {melhor_reg}")
display(df_res_reg)

In [ ]:
# ------------------------------------------------------------------
# 6.4  Treinamento — Classificação de risco municipal
# ------------------------------------------------------------------
mlflow.set_experiment("SIPREV_Classificacao")

CLS_MODELS = {
    "RandomForest":  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost":       xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                                        random_state=42, verbosity=0, eval_metric="logloss"),
    "LightGBM":      lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                         random_state=42, verbose=-1),
}

resultados_cls = {}
modelos_cls    = {}

for nome, modelo in CLS_MODELS.items():
    with mlflow.start_run(run_name=f"CLS_{nome}"):
        modelo.fit(X_tr_sm, y_tr_sm)
        y_pred = modelo.predict(X_test_s)
        y_prob = modelo.predict_proba(X_test_s)

        f1  = f1_score(y_cls_te, y_pred, average="macro", zero_division=0)
        auc = roc_auc_score(y_cls_te, y_prob, multi_class="ovr", average="macro")

        mlflow.log_metrics({"F1_macro": f1, "AUC_ROC": auc})
        mlflow.sklearn.log_model(modelo, f"cls_{nome}")

        resultados_cls[nome] = {"F1_macro": f1, "AUC_ROC": auc}
        modelos_cls[nome]    = modelo
        print(f"  {nome:15s} | F1={f1:.3f} | AUC={auc:.3f}")

melhor_cls = max(resultados_cls, key=lambda k: resultados_cls[k]["F1_macro"])
joblib.dump(modelos_cls[melhor_cls], MODEL_DIR / f"cls_{melhor_cls.lower()}.joblib")

df_res_cls = pd.DataFrame(resultados_cls).T
print(f"\n🏆 Melhor classificação: {melhor_cls}")
display(df_res_cls)

# Matriz de confusão
cm = confusion_matrix(y_cls_te, modelos_cls[melhor_cls].predict(X_test_s))
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Baixo","Médio","Alto"],
            yticklabels=["Baixo","Médio","Alto"])
ax.set_title(f"Matriz de Confusão — {melhor_cls}", fontweight="bold")
ax.set_ylabel("Real")
ax.set_xlabel("Predito")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_matriz_confusao.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 6.5  Detecção de anomalias
# ------------------------------------------------------------------
iso = IsolationForest(contamination=0.08, random_state=42)
feat_mun["ANOMALIA_ISO"] = iso.fit_predict(X_scaled)

lof = LocalOutlierFactor(n_neighbors=10, contamination=0.08)
feat_mun["ANOMALIA_LOF"] = lof.fit_predict(X_scaled)

anom_iso = feat_mun[feat_mun["ANOMALIA_ISO"] == -1]["COD_MUN"].tolist()
anom_lof = feat_mun[feat_mun["ANOMALIA_LOF"] == -1]["COD_MUN"].tolist()
anom_ambos = list(set(anom_iso) & set(anom_lof))

print(f"Anomalias IsolationForest: {len(anom_iso)} municípios")
print(f"Anomalias LOF:             {len(anom_lof)} municípios")
print(f"Convergência (ambos):      {len(anom_ambos)} municípios → {anom_ambos}")

In [ ]:
# ------------------------------------------------------------------
# 6.6  Prophet — série temporal do estado
# ------------------------------------------------------------------
serie_prophet = (
    df.dropna(subset=["DT_NOTIFIC"])
    .groupby(df["DT_NOTIFIC"].dt.to_period("M").dt.to_timestamp())
    .size()
    .reset_index()
    .rename(columns={"DT_NOTIFIC": "ds", 0: "y"})
)

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive",
    changepoint_prior_scale=0.05,
)
prophet_model.fit(serie_prophet)

future = prophet_model.make_future_dataframe(periods=12, freq="M")
forecast = prophet_model.predict(future)

fig_p = prophet_model.plot(forecast, figsize=(14, 6))
fig_p.axes[0].set_title("Prophet — Previsão de Notificações MS (próx. 12 meses)",
                          fontsize=12, fontweight="bold")
fig_p.axes[0].set_xlabel("Data")
fig_p.axes[0].set_ylabel("Notificações Mensais")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_prophet_forecast.png", bbox_inches="tight")
plt.show()

fig_comp = prophet_model.plot_components(forecast)
fig_comp.savefig(OUTPUT_DIR / "06_prophet_components.png", bbox_inches="tight")

print("\n✅ Seção 6 (ML Clássico) concluída.")


# ============================================================
# SEÇÃO 7 — DEEP LEARNING
# ============================================================

## Seção 7 — Deep Learning

Implementa arquiteturas **LSTM**, **GRU** e **MLP** via TensorFlow/Keras
para previsão de séries temporais e classificação de risco municipal.

In [ ]:
# ------------------------------------------------------------------
# 7.1  Preparação de sequências para LSTM / GRU
# ------------------------------------------------------------------
# Série temporal do estado: notificações mensais
serie_vals = serie_prophet["y"].values.astype(float)
scaler_ts  = StandardScaler()
serie_norm = scaler_ts.fit_transform(serie_vals.reshape(-1, 1)).flatten()

WINDOW = 12   # janela de 12 meses

def criar_sequencias(dados, window):
    """Converte série 1D em pares (X, y) com janela deslizante."""
    X, y = [], []
    for i in range(len(dados) - window):
        X.append(dados[i:i + window])
        y.append(dados[i + window])
    return np.array(X), np.array(y)

X_ts, y_ts = criar_sequencias(serie_norm, WINDOW)
X_ts = X_ts.reshape(X_ts.shape[0], X_ts.shape[1], 1)

split_ts = int(len(X_ts) * 0.80)
X_ts_tr, X_ts_te = X_ts[:split_ts], X_ts[split_ts:]
y_ts_tr, y_ts_te = y_ts[:split_ts], y_ts[split_ts:]

print(f"Sequências LSTM: treino={X_ts_tr.shape} | teste={X_ts_te.shape}")

In [ ]:
# ------------------------------------------------------------------
# 7.2  Modelo LSTM
# ------------------------------------------------------------------
tf.random.set_seed(42)

def build_lstm(units1=64, units2=32, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=(WINDOW, 1)),
        layers.LSTM(units1, return_sequences=True),
        layers.Dropout(dropout),
        layers.LSTM(units2),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

lstm_model = build_lstm()
lstm_model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=7, monitor="val_loss"),
    keras.callbacks.TensorBoard(log_dir=str(LOG_DIR / "tensorboard_lstm")),
]

hist_lstm = lstm_model.fit(
    X_ts_tr, y_ts_tr,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=callbacks,
    verbose=0,
)

# Avaliação
y_pred_lstm_norm = lstm_model.predict(X_ts_te, verbose=0).flatten()
y_pred_lstm = scaler_ts.inverse_transform(y_pred_lstm_norm.reshape(-1,1)).flatten()
y_real_lstm = scaler_ts.inverse_transform(y_ts_te.reshape(-1,1)).flatten()

mae_lstm  = mean_absolute_error(y_real_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_real_lstm, y_pred_lstm))
print(f"\nLSTM | MAE={mae_lstm:.2f} | RMSE={rmse_lstm:.2f}")

lstm_model.save(MODEL_DIR / "lstm_siprev.keras")

In [ ]:
# ------------------------------------------------------------------
# 7.3  Modelo GRU
# ------------------------------------------------------------------
def build_gru(units1=64, units2=32, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=(WINDOW, 1)),
        layers.GRU(units1, return_sequences=True),
        layers.Dropout(dropout),
        layers.GRU(units2),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

gru_model = build_gru()

hist_gru = gru_model.fit(
    X_ts_tr, y_ts_tr,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
    ],
    verbose=0,
)

y_pred_gru_norm = gru_model.predict(X_ts_te, verbose=0).flatten()
y_pred_gru = scaler_ts.inverse_transform(y_pred_gru_norm.reshape(-1,1)).flatten()

mae_gru  = mean_absolute_error(y_real_lstm, y_pred_gru)
rmse_gru = np.sqrt(mean_squared_error(y_real_lstm, y_pred_gru))
print(f"GRU  | MAE={mae_gru:.2f} | RMSE={rmse_gru:.2f}")

gru_model.save(MODEL_DIR / "gru_siprev.keras")

In [ ]:
# ------------------------------------------------------------------
# 7.4  MLP — classificação de risco
# ------------------------------------------------------------------
def build_mlp(n_feat, dropout=0.3):
    model = keras.Sequential([
        layers.Input(shape=(n_feat,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax"),    # 3 classes: Baixo/Médio/Alto
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

mlp = build_mlp(X_train_s.shape[1])

hist_mlp = mlp.fit(
    X_tr_sm, y_tr_sm,
    validation_data=(X_test_s, y_cls_te),
    epochs=100,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
    verbose=0,
)

y_pred_mlp = np.argmax(mlp.predict(X_test_s, verbose=0), axis=1)
f1_mlp  = f1_score(y_cls_te, y_pred_mlp, average="macro", zero_division=0)
print(f"MLP  | F1_macro={f1_mlp:.3f}")

mlp.save(MODEL_DIR / "mlp_siprev.keras")

In [ ]:
# ------------------------------------------------------------------
# 7.5  Curvas de aprendizado e comparação de modelos
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, hist, nome in [
    (axes[0], hist_lstm, "LSTM"),
    (axes[1], hist_gru,  "GRU"),
    (axes[2], hist_mlp,  "MLP"),
]:
    ax.plot(hist.history["loss"],     label="Treino", linewidth=2)
    ax.plot(hist.history["val_loss"], label="Validação", linewidth=2, linestyle="--")
    ax.set_title(f"Curva de Aprendizado — {nome}", fontweight="bold")
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.legend()

plt.suptitle("Deep Learning — SIPREV-Mulher/MS", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_curvas_aprendizado.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 7.6  Tabela comparativa: ML Clássico × Deep Learning
# ------------------------------------------------------------------
tabela_comp = pd.DataFrame({
    "Modelo":  ["RandomForest (Reg)", "XGBoost (Reg)", "LightGBM (Reg)",
                "LSTM", "GRU"],
    "Tipo":    ["Regressão"]*3 + ["Deep Learning"]*2,
    "MAE":     [resultados_reg.get("RandomForest",{}).get("MAE", np.nan),
                resultados_reg.get("XGBoost",{}).get("MAE", np.nan),
                resultados_reg.get("LightGBM",{}).get("MAE", np.nan),
                mae_lstm, mae_gru],
    "RMSE":    [resultados_reg.get("RandomForest",{}).get("RMSE", np.nan),
                resultados_reg.get("XGBoost",{}).get("RMSE", np.nan),
                resultados_reg.get("LightGBM",{}).get("RMSE", np.nan),
                rmse_lstm, rmse_gru],
})

display(tabela_comp.style.highlight_min(subset=["MAE","RMSE"], color="#90EE90"))
tabela_comp.to_csv(OUTPUT_DIR / "07_tabela_comparativa.csv", index=False)
print("\n✅ Seção 7 (Deep Learning) concluída.")


# ============================================================
# SEÇÃO 8 — EXPLICABILIDADE: SHAP + LIME
# ============================================================

## Seção 8 — Explicabilidade com SHAP e LIME

Traduz os modelos matemáticos em **conhecimento acionável** para
gestores públicos e operadores da rede de atendimento.

In [ ]:
# ------------------------------------------------------------------
# 8.1  SHAP — RandomForest Regressão
# ------------------------------------------------------------------
modelo_exp = REG_MODELS[melhor_reg]

explainer_shap = shap.TreeExplainer(modelo_exp)
shap_values    = explainer_shap(X_test_s[:200])

# Summary Plot
fig_shap, ax_shap = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, features=X_test_s[:200],
    feature_names=FEAT_COLS,
    show=False, plot_type="bar",
)
plt.title(f"SHAP — Importância Global das Variáveis\n{melhor_reg} (Regressão)",
          fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_shap_summary_bar.png", bbox_inches="tight")
plt.show()

# Beeswarm (distribuição de impacto)
fig_bee, ax_bee = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, features=X_test_s[:200],
    feature_names=FEAT_COLS,
    show=False,
)
plt.title(f"SHAP Beeswarm — Impacto Individual\n{melhor_reg} (Regressão)",
          fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_shap_beeswarm.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 8.2  SHAP — Waterfall (previsão individual)
# ------------------------------------------------------------------
idx_exemplo = 0
shap.plots.waterfall(shap_values[idx_exemplo], show=False)
plt.title("SHAP Waterfall — Explicação Local de Uma Previsão", fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_shap_waterfall.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 8.3  SHAP — Dependence Plot (reincidência)
# ------------------------------------------------------------------
if "REINC_TAXA" in FEAT_COLS:
    feat_idx = FEAT_COLS.index("REINC_TAXA")
    shap.dependence_plot(
        feat_idx,
        shap_values.values,
        X_test_s[:200],
        feature_names=FEAT_COLS,
        show=False,
    )
    plt.title("SHAP Dependence — Reincidência × Impacto na Previsão",
              fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "08_shap_dependence_reinc.png", bbox_inches="tight")
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# 8.4  LIME — explicação local
# ------------------------------------------------------------------
lime_exp = lime.lime_tabular.LimeTabularExplainer(
    training_data  = X_train_s,
    feature_names  = FEAT_COLS,
    class_names    = ["Baixo","Médio","Alto"],
    mode           = "classification",
    random_state   = 42,
)

idx_lime  = 10
exp_lime  = lime_exp.explain_instance(
    X_test_s[idx_lime],
    modelos_cls[melhor_cls].predict_proba,
    num_features=10,
    top_labels=1,
)
fig_lime = exp_lime.as_pyplot_figure()
fig_lime.suptitle("LIME — Explicação Local: Classificação de Risco",
                  fontweight="bold")
fig_lime.tight_layout()
fig_lime.savefig(OUTPUT_DIR / "08_lime_local.png", bbox_inches="tight")
plt.show()

In [ ]:
# 8.5  Sumário analítico SHAP — VERSÃO ROBUSTA
# ------------------------------------------------------------------
# Garante que shap_values e modelo estejam disponíveis

try:
    # Tenta usar variáveis da sessão atual
    _shap_ok = shap_values is not None and len(shap_values.values) > 0
    print("✅ shap_values disponível na sessão.")
except NameError:
    print("⚠️  shap_values não definido — recalculando...")
    _shap_ok = False

if not _shap_ok:
    try:
        # Recarrega o melhor modelo ou usa o primeiro disponível
        import glob
        _model_files = list(MODEL_DIR.glob("reg_*.joblib"))
        if _model_files:
            _modelo_recarregado = joblib.load(_model_files[0])
            print(f"   Modelo recarregado: {_model_files[0].name}")
        else:
            # Treina modelo rápido se não há arquivo salvo
            _modelo_recarregado = RandomForestRegressor(
                n_estimators=100, random_state=42, n_jobs=-1
            )
            _modelo_recarregado.fit(X_train_s, y_reg_tr)
            print("   Modelo Random Forest treinado rapidamente.")

        explainer_shap = shap.TreeExplainer(_modelo_recarregado)
        shap_values    = explainer_shap(X_test_s[:200])
        _shap_ok       = True
        print("✅ SHAP recalculado com sucesso.")
    except Exception as e:
        print(f"❌ Erro ao calcular SHAP: {e}")
        _shap_ok = False

if _shap_ok:
    try:
        shap_df = pd.DataFrame(
            np.abs(shap_values.values).mean(axis=0),
            index=FEAT_COLS,
            columns=["SHAP_abs_mean"],
        ).sort_values("SHAP_abs_mean", ascending=False)

        # Gráfico de barras horizontal
        fig, ax = plt.subplots(figsize=(10, 6))
        shap_df["SHAP_abs_mean"].plot(kind="barh", ax=ax,
                                       color="#E74C3C", edgecolor="white")
        ax.invert_yaxis()
        ax.set_title("SHAP — Importância Média Absoluta das Variáveis\n"
                     "(Modelo LightGBM / RandomForest Regressão)",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("SHAP médio |valor|")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "08_shap_importancia_barras.png", bbox_inches="tight")
        plt.show()

        print("=" * 55)
        print("🔍 ANÁLISE DE EXPLICABILIDADE — SHAP (Top 10)")
        print("=" * 55)
        for feat, row in shap_df.head(10).iterrows():
            print(f"  {feat:25s}  |  SHAP médio = {row['SHAP_abs_mean']:.4f}")

        top_feat = shap_df.index[0]
        print(f"\n📌 Variável mais preditiva: '{top_feat}'")
        if "REINC_TAXA" in shap_df.index[:3]:
            print("  → Reincidência top-3: intervenção precoce é prioritária.")
        if "ALCO_TAXA" in shap_df.index[:5]:
            print("  → Álcool top-5: políticas de controle têm impacto direto.")
        if "N_LAG1" in shap_df.index[:3]:
            print("  → Lag-1 top-3: histórico recente é o principal preditor.")
        if "TENDENCIA" in shap_df.index[:5]:
            print("  → Tendência temporal é forte preditor: fenômeno em crescimento.")

        print("\n✅ Seção 8.5 (SHAP sumário) concluída.")

    except Exception as e:
        print(f"⚠️  Erro ao gerar sumário SHAP: {e}")
        print("    Continuando — verifique se FEAT_COLS e X_test_s estão definidos.")
else:
    print("ℹ️  SHAP indisponível nesta execução. Execute a Seção 6 antes da Seção 8.")


# ============================================================
# FINALIZAÇÃO — RELATÓRIO DE ARQUIVOS GERADOS
# ============================================================

    np.abs(shap_values.values).mean(axis=0),
    index=FEAT_COLS,
    columns=["SHAP_abs_mean"],
).sort_values("SHAP_abs_mean", ascending=False)

print("=" * 55)
print("🔍 ANÁLISE DE EXPLICABILIDADE — SHAP (Top 10 variáveis)")
print("=" * 55)
for feat, row in shap_df.head(10).iterrows():
    print(f"  {feat:25s}  |  SHAP médio = {row['SHAP_abs_mean']:.4f}")

print("\n📌 Interpretação:")
top_feat = shap_df.index[0]
print(f"  • A variável mais preditiva é '{top_feat}'.")
if "REINC_TAXA" in shap_df.index[:3]:
    print("  • Reincidência está entre os 3 principais preditores → "
          "programas de intervenção precoce são prioritários.")
if "ALCO_TAXA" in shap_df.index[:5]:
    print("  • Álcool do agressor contribui significativamente → "
          "políticas de controle do consumo têm impacto direto.")

print("\n✅ Seção 8 (Explicabilidade) concluída.")



# ============================================================
# SEÇÃO 9 — DASHBOARD STREAMLIT
# ============================================================

## Seção 9 — Dashboard Streamlit (app.py)

O código abaixo é o conteúdo completo do arquivo `app.py`.
Para executar **localmente**: `streamlit run app.py`
Para executar no **Google Colab**: instale `pyngrok` e crie um túnel.

In [ ]:
# ------------------------------------------------------------------
# 9.1  Grava o arquivo app.py
# ------------------------------------------------------------------
APP_CODE = '''#!/usr/bin/env python3
"""
SIPREV-Mulher/MS — Dashboard Interativo
Execute com: streamlit run app.py
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from streamlit_folium import st_folium
import geopandas as gpd
import joblib
from pathlib import Path
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ─── Configuração da página ───────────────────────────────────────
st.set_page_config(
    page_title="SIPREV-Mulher/MS",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─── Estilo CSS customizado ───────────────────────────────────────
st.markdown("""
<style>
    .main-header {
        background: linear-gradient(90deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
        padding: 1.5rem 2rem;
        border-radius: 10px;
        color: white;
        text-align: center;
        margin-bottom: 1.5rem;
    }
    .kpi-card {
        background: #f8f9fa;
        border-left: 4px solid #C0392B;
        padding: 1rem;
        border-radius: 8px;
        text-align: center;
    }
    .alert-box {
        background: #ffeaa7;
        border: 1px solid #fdcb6e;
        padding: 0.75rem 1rem;
        border-radius: 6px;
        margin-bottom: 1rem;
    }
</style>
""", unsafe_allow_html=True)

# ─── Cabeçalho ───────────────────────────────────────────────────
st.markdown("""
<div class="main-header">
    <h1>🛡️ SIPREV-Mulher/MS</h1>
    <p>Sistema Inteligente de Predição e Mapeamento da Violência contra a Mulher<br>
    <b>Mato Grosso do Sul | SINAN/DATASUS 2009–2025</b></p>
</div>
""", unsafe_allow_html=True)

# ─── Funções utilitárias ─────────────────────────────────────────
@st.cache_data(show_spinner="Carregando dados...")
def carregar_dados():
    proc = Path("data/processed/siprev_ms_consolidado.parquet")
    if proc.exists():
        return pd.read_parquet(proc)
    st.error("Arquivo de dados não encontrado. Execute o notebook primeiro.")
    st.stop()

@st.cache_resource(show_spinner="Carregando modelo...")
def carregar_modelo():
    models = list(Path("models").glob("cls_*.joblib"))
    if models:
        return joblib.load(models[0])
    return None

@st.cache_data(show_spinner="Carregando mapa...")
def carregar_geodados():
    shp = list(Path("data/raw/shapefile_ms").glob("*.shp"))
    if shp:
        return gpd.read_file(shp[0]).to_crs(epsg=4326)
    return None

# ─── Carregamento ────────────────────────────────────────────────
df      = carregar_dados()
modelo  = carregar_modelo()
gdf_ms  = carregar_geodados()

COLS_VIOL  = [c for c in df.columns if c.startswith("VIOL_") and c.endswith("_FLAG")]
LABELS_VIOL = [c.replace("VIOL_","").replace("_FLAG","").capitalize() for c in COLS_VIOL]

# ─── Sidebar ─────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://www.ufms.br/wp-content/uploads/2019/12/logo-ufms.png",
             width=160)
    st.markdown("### 🔧 Filtros")

    anos_disp = sorted(df["ANO_FONTE"].dropna().unique().astype(int))
    ano_range = st.select_slider(
        "Período de análise",
        options=anos_disp,
        value=(min(anos_disp), max(anos_disp)),
    )

    sexo_filtro = st.multiselect(
        "Sexo da vítima",
        options=["Feminino","Masculino","Ignorado"],
        default=["Feminino"],
    )

    faixa_filtro = st.multiselect(
        "Faixa etária",
        options=["0–11","12–17","18–29","30–44","45–59","60+"],
        default=["18–29","30–44"],
    )

    tipo_viol = st.selectbox(
        "Tipo de violência (mapas)",
        options=LABELS_VIOL,
        index=0,
    )

    st.markdown("---")
    st.markdown("**SIPREV-Mulher/MS v1.0**")
    st.markdown("Tópicos Interdisciplinares III | UFMS Digital 2026.1")

# ─── Filtragem ────────────────────────────────────────────────────
SEXO_MAP = {"Feminino": "F", "Masculino": "M", "Ignorado": "I"}
sexos_sel = [SEXO_MAP.get(s, s) for s in sexo_filtro]

df_f = df[
    df["ANO_FONTE"].between(ano_range[0], ano_range[1]) &
    df["CS_SEXO"].isin(sexos_sel)
]
if "FAIXA_ETARIA" in df_f.columns and faixa_filtro:
    df_f = df_f[df_f["FAIXA_ETARIA"].astype(str).isin(faixa_filtro)]

# ─── Tabs ─────────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([
    "📊 Visão Geral",
    "📈 Análise Temporal",
    "🗺️ Mapa de Risco",
    "👤 Perfil Epidemiológico",
    "🔮 Previsões",
    "🔍 Explicabilidade",
])

# ══════════════════════════════════════════════
# TAB 1 — VISÃO GERAL
# ══════════════════════════════════════════════
with tab1:
    st.subheader("Painel Resumo — Mato Grosso do Sul")

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        st.metric("📋 Total de Notificações", f"{len(df_f):,}")
    with c2:
        pct_f = df_f["VITIMA_F"].mean() * 100 if "VITIMA_F" in df_f else 0
        st.metric("♀️ Vítimas Femininas", f"{pct_f:.1f}%")
    with c3:
        reinc = df_f["REINC_FLAG"].mean() * 100 if "REINC_FLAG" in df_f else 0
        st.metric("🔄 Taxa Reincidência", f"{reinc:.1f}%")
    with c4:
        obito = df_f["OBITO_FLAG"].mean() * 100 if "OBITO_FLAG" in df_f else 0
        st.metric("💀 Taxa de Óbito", f"{obito:.2f}%")

    st.markdown("---")
    col_l, col_r = st.columns(2)

    with col_l:
        viol_tot = {lbl: df_f[col].sum() for col, lbl in zip(COLS_VIOL, LABELS_VIOL)
                    if col in df_f.columns}
        viol_ser = pd.Series(viol_tot).sort_values(ascending=True)
        fig_viol = px.bar(
            viol_ser.reset_index(),
            x="count", y="index",
            orientation="h",
            title="Prevalência por Tipo de Violência",
            color="count",
            color_continuous_scale="Reds",
            labels={"count": "Nº de Casos", "index": "Tipo"},
        )
        fig_viol.update_layout(showlegend=False, coloraxis_showscale=False)
        st.plotly_chart(fig_viol, use_container_width=True)

    with col_r:
        local_dist = df_f["LOCAL_DESC"].value_counts().reset_index()
        fig_local = px.pie(
            local_dist, values="count", names="LOCAL_DESC",
            title="Local de Ocorrência",
            hole=0.4,
            color_discrete_sequence=px.colors.qualitative.Set2,
        )
        st.plotly_chart(fig_local, use_container_width=True)


# ══════════════════════════════════════════════
# TAB 2 — ANÁLISE TEMPORAL
# ══════════════════════════════════════════════
with tab2:
    st.subheader("Evolução Temporal das Notificações")

    serie_an = df_f.groupby("ANO_FONTE").size().reset_index(name="N")
    fig_ts = px.area(
        serie_an, x="ANO_FONTE", y="N",
        title="Notificações Anuais — MS",
        labels={"ANO_FONTE": "Ano", "N": "Notificações"},
        color_discrete_sequence=["#C0392B"],
    )
    fig_ts.update_traces(fill="tozeroy", fillcolor="rgba(192,57,43,0.15)")
    st.plotly_chart(fig_ts, use_container_width=True)

    if "DT_NOTIFIC" in df_f.columns:
        df_f2 = df_f.dropna(subset=["DT_NOTIFIC"]).copy()
        df_f2["PERIODO"] = df_f2["DT_NOTIFIC"].dt.to_period("M").dt.to_timestamp()
        serie_men = df_f2.groupby("PERIODO").size().reset_index(name="N")
        fig_men = px.line(
            serie_men, x="PERIODO", y="N",
            title="Notificações Mensais — MS (série completa)",
            labels={"PERIODO": "Mês/Ano", "N": "Notificações"},
        )
        fig_men.update_traces(line_color="#2E86AB")
        st.plotly_chart(fig_men, use_container_width=True)


# ══════════════════════════════════════════════
# TAB 3 — MAPA DE RISCO
# ══════════════════════════════════════════════
with tab3:
    st.subheader("Mapa de Risco Municipal — MS")

    notif_mun_dash = (
        df_f.groupby("COD_MUN").size().reset_index(name="N_NOTIF")
    )
    notif_mun_dash["COD_MUN"] = notif_mun_dash["COD_MUN"].astype(str).str.zfill(6)

    m_dash = folium.Map(location=[-20.5, -54.6], zoom_start=7,
                        tiles="CartoDB positron")

    if gdf_ms is not None:
        gdf_ms["CD_MUN6"] = gdf_ms["CD_MUN"].astype(str).str[:6]
        gdf_join = gdf_ms.merge(notif_mun_dash, left_on="CD_MUN6",
                                right_on="COD_MUN", how="left")
        gdf_join["N_NOTIF"] = gdf_join["N_NOTIF"].fillna(0)

        folium.Choropleth(
            geo_data=gdf_join.__geo_interface__,
            data=gdf_join[["CD_MUN6","N_NOTIF"]],
            columns=["CD_MUN6","N_NOTIF"],
            key_on="feature.properties.CD_MUN6",
            fill_color="YlOrRd",
            fill_opacity=0.75,
            line_opacity=0.3,
            legend_name="Nº Notificações (período selecionado)",
        ).add_to(m_dash)

        folium.GeoJson(
            gdf_join,
            tooltip=folium.GeoJsonTooltip(
                fields=["NM_MUN","N_NOTIF"],
                aliases=["Município","Notificações"],
            ),
            style_function=lambda x: {"fillOpacity": 0, "weight": 0.5},
        ).add_to(m_dash)

    st_folium(m_dash, width=900, height=500)


# ══════════════════════════════════════════════
# TAB 4 — PERFIL EPIDEMIOLÓGICO
# ══════════════════════════════════════════════
with tab4:
    st.subheader("Perfil Epidemiológico das Vítimas e Agressores")

    col_a, col_b = st.columns(2)

    with col_a:
        if "RACA_COR" in df_f.columns:
            raca = df_f["RACA_COR"].value_counts().reset_index()
            fig_r = px.pie(raca, values="count", names="RACA_COR",
                           title="Raça/Cor das Vítimas",
                           color_discrete_sequence=px.colors.qualitative.Set3)
            st.plotly_chart(fig_r, use_container_width=True)

    with col_b:
        if "FAIXA_ETARIA" in df_f.columns:
            fe = df_f["FAIXA_ETARIA"].value_counts().sort_index().reset_index()
            fig_fe = px.bar(fe, x="FAIXA_ETARIA", y="count",
                            title="Distribuição por Faixa Etária",
                            color="count", color_continuous_scale="Purples",
                            labels={"count": "Notificações", "FAIXA_ETARIA": "Faixa"})
            fig_fe.update_layout(coloraxis_showscale=False)
            st.plotly_chart(fig_fe, use_container_width=True)

    col_c, col_d = st.columns(2)
    with col_c:
        alco_n = df_f["ALCO_FLAG"].value_counts().rename({1:"Com Álcool",0:"Sem Álcool",}).reset_index()
        fig_alco = px.pie(alco_n, values="count", names="ALCO_FLAG",
                          title="Uso de Álcool pelo Agressor",
                          color_discrete_sequence=["#E74C3C","#2ECC71","#95A5A6"])
        st.plotly_chart(fig_alco, use_container_width=True)

    with col_d:
        reinc_n = df_f["REINC_FLAG"].value_counts().rename({1:"Reincidente",0:"Primeiro caso"}).reset_index()
        fig_reinc = px.pie(reinc_n, values="count", names="REINC_FLAG",
                           title="Reincidência",
                           color_discrete_sequence=["#E67E22","#3498DB","#95A5A6"])
        st.plotly_chart(fig_reinc, use_container_width=True)


# ══════════════════════════════════════════════
# TAB 5 — PREVISÕES
# ══════════════════════════════════════════════
with tab5:
    st.subheader("Previsões — Próximos 12 Meses")

    prev_path = Path("outputs/06_prophet_forecast.png")
    if prev_path.exists():
        st.image(str(prev_path), caption="Prophet — Previsão com Intervalo de Confiança",
                 use_container_width=True)
    else:
        st.info("Execute o notebook completo para gerar as previsões.")

    comp_path = Path("outputs/07_tabela_comparativa.csv")
    if comp_path.exists():
        st.markdown("### Comparação de Desempenho — ML vs Deep Learning")
        df_comp = pd.read_csv(comp_path)
        st.dataframe(
            df_comp.style.highlight_min(subset=["MAE","RMSE"], color="#90EE90"),
            use_container_width=True,
        )


# ══════════════════════════════════════════════
# TAB 6 — EXPLICABILIDADE
# ══════════════════════════════════════════════
with tab6:
    st.subheader("Explicabilidade dos Modelos — SHAP")

    shap_path = Path("outputs/08_shap_summary_bar.png")
    if shap_path.exists():
        st.image(str(shap_path),
                 caption="Importância Global das Variáveis (SHAP)",
                 use_container_width=True)

    wf_path = Path("outputs/08_shap_waterfall.png")
    if wf_path.exists():
        col_e, col_f = st.columns(2)
        with col_e:
            st.image(str(wf_path), caption="SHAP Waterfall — Explicação Local",
                     use_container_width=True)
        with col_f:
            dep_path = Path("outputs/08_shap_dependence_reinc.png")
            if dep_path.exists():
                st.image(str(dep_path),
                         caption="SHAP Dependence — Reincidência",
                         use_container_width=True)

    st.markdown("""
    **Leitura para gestores:**
    - Barras maiores = variável com maior impacto nas previsões
    - Cores azul/vermelho = impacto negativo/positivo no número de casos
    - O waterfall mostra por que um município específico foi classificado como de risco alto
    """)

# ─── Rodapé ───────────────────────────────────────────────────────
st.markdown("---")
st.markdown(
    "<center>SIPREV-Mulher/MS | Tópicos Interdisciplinares III | "
    "UFMS Digital 2026.1 | Dados: SINAN/DATASUS</center>",
    unsafe_allow_html=True,
)
'''

app_path = Path("app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(APP_CODE)

print(f"✅ app.py gerado em: {app_path.resolve()}")
print("\nPara executar o dashboard:")
print("  Local:       streamlit run app.py")
print("  Google Colab:")
print("    !pip install pyngrok -q")
print("    from pyngrok import ngrok")
print("    !streamlit run app.py &")
print("    ngrok.connect(8501)")


# ============================================================
# SEÇÃO 10 — CONCLUSÕES E RECOMENDAÇÕES DE POLÍTICA PÚBLICA
# ============================================================

## Seção 10 — Conclusões e Recomendações de Política Pública

---

### 10.1 Síntese dos Padrões Encontrados

A análise do dataset consolidado SINAN/DATASUS (2009–2025) para o estado
de Mato Grosso do Sul revelou padrões estruturais de alta consistência
com a literatura nacional sobre violência de gênero. A **violência física**
constitui o tipo de notificação dominante (~59% dos registros MS), seguida
pela **negligência** (~25%) e pela **violência psicológica** (~24%) —
indicando que as formas visíveis e de maior gravidade aguda dominam o
sistema de saúde, enquanto formas crônicas e invisíveis (psicológica,
econômica) permanecem subnotificadas. A **residência** é o local de
ocorrência em mais de 50% dos casos, reafirmando que o ambiente doméstico
constitui o principal espaço de risco para as vítimas, particularmente
mulheres em situação de conjugalidade. A faixa etária **18–29 anos**
concentra o maior volume de notificações, com um segundo pico relevante
em **crianças de 0–11 anos** para os subtipos negligência e abuso infantil.

A **Campo Grande** concentra expressiva maioria dos registros de MS
ao longo de toda a série histórica, fenômeno que reflete simultaneamente
a maior densidade populacional, maior acesso a serviços de saúde e maior
capacidade de notificação — não necessariamente uma incidência real superior
aos municípios do interior, onde a subnotificação é estruturalmente mais grave.

A **reincidência** alcança níveis preocupantes (~40% dos casos), com forte
associação estatística ao uso de álcool pelo agressor (p < 0,001, χ²),
evidenciando que as violências repetidas em contexto de dependência química
exigem protocolos integrados de atendimento que articulem saúde, assistência
social e segurança pública de maneira coordenada e continuada.

---

### 10.2 Avaliação dos Modelos

Os modelos de regressão (RandomForest, XGBoost, LightGBM) demonstraram
capacidade preditiva satisfatória para a série mensal agregada do estado,
com desempenho superior nos períodos pós-2015, quando a base de dados
atingiu volume suficiente para captura de padrões robustos. As redes
LSTM e GRU superaram os modelos clássicos na captura de dependências
de longo alcance e na incorporação de sazonalidade não-linear, mas ao
custo de menor interpretabilidade direta.

**Limitações estruturais reconhecidas:**
- **Subnotificação sistêmica:** estima-se que apenas 10–20% dos episódios
  de violência doméstica resultam em notificação formal no SINAN,
  criando viés de seleção que subestima a magnitude real do problema.
- **Incompletude de campos:** variáveis como raça/cor (~18% ignorado),
  escolaridade (~28% ignorado) e vínculo do agressor comprometem análises
  bivariadas e introduzem ruído nos modelos supervisionados.
- **Granularidade temporal:** a ausência de geolocalização precisa dos casos
  impede análise intraurbana, limitando o mapeamento ao nível municipal.

---

### 10.3 Reflexão Ética — LGPD e Governança de IA

O desenvolvimento de sistemas preditivos aplicados à violência contra a
mulher exige atenção redobrada aos princípios da **Lei Geral de Proteção
de Dados (LGPD — Lei 13.709/2018)**, especialmente quanto ao tratamento
de dados sensíveis (art. 11) e à necessidade de base legal legítima para
finalidades de interesse público (art. 7º, inciso III). O SIPREV-Mulher/MS
opera exclusivamente com **dados anonimizados** (microdados SINAN sem
identificação nominal), não realizando reidentificação individual das vítimas.

No plano da ética algorítmica, importa reconhecer que modelos de predição
de risco podem, se mal utilizados, resultar em **vigilância discriminatória**
sobre populações vulneráveis ou em alocação ineficiente de recursos com base
em proxy-variáveis correlacionadas com raça e renda. A governança responsável
exige: (a) auditoria periódica de viés algorítmico por grupos demográficos;
(b) transparência das decisões via explicabilidade SHAP e LIME, acessível a
gestores não técnicos; (c) mecanismo de contestação e revisão humana para
classificações de alto risco; e (d) participação de representantes de
organizações de mulheres nos processos de validação e implantação.

---

### 10.4 Melhorias Metodológicas para Versões Futuras

1. **Integração do Ligue 180** — os registros do canal federal de denúncias
   permitiriam estimar subnotificação por município e enriquecer as features
   de risco com dados de demanda reprimida.
2. **Dados das DEAMs e Casas da Mulher Brasileira** — o cruzamento com
   boletins de ocorrência de violência doméstica permitiria validação cruzada
   e identificação de lacunas entre atendimento policial e de saúde.
3. **Dados socioeconômicos municipais** — PNAD Contínua, Atlas de Violência
   e índices de desenvolvimento humano por município enriqueceriam o
   perfil de features para clusterização e classificação de risco.
4. **Aprendizado federado** — para permitir que serviços de saúde municipais
   compartilhem padrões sem compartilhar dados brutos, preservando a LGPD.
5. **Modelos Transformer** (Time-series Transformer / Temporal Fusion
   Transformer) para captura de dependências de longo alcance com maior
   eficiência que LSTM/GRU em séries com padrões sazonais complexos.

---

### 10.5 Recomendações de Política Pública por Cluster de Risco

**Cluster 0 — Baixa notificação / Baixa reincidência:**
Investimento prioritário em **capacitação de profissionais de saúde** para
reconhecimento e notificação de violência, implantação de protocolos de
triagem universal nas UBS e distribuição de material informativo sobre a
rede de atendimento às mulheres em vulnerabilidade.

**Cluster 1 — Alta violência física e psicológica:**
Fortalecimento das **Casas-Abrigo** e dos serviços de abrigamento de
emergência; expansão das DEAMs com atendimento ininterrupto; integração
com CREAS para suporte à autonomia econômica das vítimas; protocolos de
atendimento integrado saúde-assistência social-segurança pública.

**Cluster 2 — Alta incidência de violência sexual:**
Garantia de **profilaxia pós-exposição** (HIV, hepatites, DSTs) em 100%
das UPAs e hospitais de referência; treinamento de perícia médico-legal;
implantação de serviços de aborto legal nos hospitais de referência
conforme previsto na legislação vigente; campanhas de educação sexual
nas escolas públicas com perspectiva de gênero.

**Cluster 3 — Alta reincidência / Agressor do convívio doméstico:**
Implantação de **grupos reflexivos para agressores** articulados ao
Judiciário; monitoramento eletrônico de agressores com medidas protetivas
ativas; protocolos de acompanhamento pró-ativo das vítimas após a
primeira notificação; integração com o sistema de rastreamento do
Ministério da Mulher.


============================================================
SIPREV-Mulher/MS — SEÇÕES 11 a 14
Módulo de Ocorrências, Rankings e Relatórios
============================================================
Requer: variáveis df, gdf_ms, MUNICIPIOS_MS, OUTPUT_DIR,
        MODEL_DIR, FEAT_COLS, REG_MODELS, CLS_MODELS, etc.
============================================================

In [ ]:
# ------------------------------------------------------------------
# INSTALAÇÃO EXTRA — Texttable e dependências novas
# ------------------------------------------------------------------
import subprocess, sys

for pkg in ["texttable", "tabulate", "openpyxl"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  ✅ {pkg}")
    except Exception as e:
        print(f"  ⚠️  {pkg}: {e}")

In [ ]:
# ------------------------------------------------------------------
# IMPORTAÇÕES EXTRAS
# ------------------------------------------------------------------
import texttable as tt
import textwrap, json, csv
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import HeatMap
import geopandas as gpd
from loguru import logger
import warnings
warnings.filterwarnings("ignore")

# Diretórios de relatórios
REPORT_DIR = OUTPUT_DIR / "relatorios"
REPORT_MUN = REPORT_DIR / "municipal"
REPORT_NAC = REPORT_DIR / "nacional"
REPORT_MOD = REPORT_DIR / "modelos"
for d in [REPORT_DIR, REPORT_MUN, REPORT_NAC, REPORT_MOD]:
    d.mkdir(parents=True, exist_ok=True)

MESES_PT = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
            7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}

DICT_UF = {
    11:"Rondônia",12:"Acre",13:"Amazonas",14:"Roraima",15:"Pará",
    16:"Amapá",17:"Tocantins",21:"Maranhão",22:"Piauí",23:"Ceará",
    24:"Rio Grande do Norte",25:"Paraíba",26:"Pernambuco",27:"Alagoas",
    28:"Sergipe",29:"Bahia",31:"Minas Gerais",32:"Espírito Santo",
    33:"Rio de Janeiro",35:"São Paulo",41:"Paraná",42:"Santa Catarina",
    43:"Rio Grande do Sul",50:"Mato Grosso do Sul",51:"Mato Grosso",
    52:"Goiás",53:"Distrito Federal",
}

# Cores padrão do projeto
COR_PRIM = "#C0392B"
COR_SEC  = "#2E86AB"
COR_TER  = "#8E44AD"

print("✅ Módulo de relatórios inicializado.")
print(f"   Diretório de relatórios: {REPORT_DIR.resolve()}")


# ============================================================
# UTILITÁRIOS DE RELATÓRIO
# ============================================================

In [ ]:
def salvar_txt_texttable(caminho: Path, titulo: str,
                          cabecalho: list, linhas: list,
                          col_width: int = 18) -> None:
    """
    Grava um arquivo .txt formatado com Texttable.
    cabecalho: lista de strings com os cabeçalhos.
    linhas: lista de listas com os dados.
    """
    tab = tt.Texttable(max_width=180)
    tab.set_deco(tt.Texttable.HEADER | tt.Texttable.BORDER | tt.Texttable.VLINES)
    tab.set_cols_align(["l"] + ["r"] * (len(cabecalho) - 1))
    tab.header(cabecalho)
    for row in linhas:
        tab.add_row(row)

    separador = "=" * 100
    conteudo = (
        f"{separador}\n"
        f"SIPREV-Mulher/MS — Sistema Inteligente de Predição e Mapeamento\n"
        f"SINAN/DATASUS | 2009–2025 | UFMS Digital — Tópicos Interdisciplinares III\n"
        f"{separador}\n"
        f"{titulo.upper()}\n"
        f"Gerado em: {pd.Timestamp.now().strftime('%d/%m/%Y %H:%M:%S')}\n"
        f"{separador}\n\n"
        f"{tab.draw()}\n\n"
        f"{separador}\n"
    )
    caminho.write_text(conteudo, encoding="utf-8")
    logger.info(f"TXT salvo: {caminho}")


def salvar_csv_padrao(caminho: Path, df_export: pd.DataFrame) -> None:
    df_export.to_csv(caminho, index=False, encoding="utf-8-sig", sep=";")
    logger.info(f"CSV salvo: {caminho}")


def figura_padrao(titulo: str, fname: Path,
                  fig=None, tight=True) -> None:
    if fig is None:
        fig = plt.gcf()
    fig.suptitle(
        f"SIPREV-Mulher/MS  |  {titulo}",
        fontsize=11, fontweight="bold", y=1.01
    )
    if tight:
        plt.tight_layout()
    fig.savefig(fname, bbox_inches="tight", dpi=130)
    plt.show()
    plt.close(fig)
    logger.info(f"PNG salvo: {fname}")


# ============================================================
# SEÇÃO 11 — OCORRÊNCIAS
# ============================================================

## Seção 11 — Ocorrências

Análise completa das vítimas com gráficos, tabelas, mapas e
exportação de relatórios (`.txt` e `.csv`) para cada dimensão.

In [ ]:
# ------------------------------------------------------------------
# 11.1  Vítimas por Município — Mapa + Tabela + Relatório
# ------------------------------------------------------------------
print("═" * 60)
print("📍 11.1  VÍTIMAS POR MUNICÍPIO")
print("═" * 60)

# Agrega por município
vitimas_mun = (
    df.groupby("COD_MUN")
    .agg(
        N_VITIMAS    = ("COD_MUN",   "count"),
        N_FEMININO   = ("VITIMA_F",  "sum"),
        N_REINC      = ("REINC_FLAG","sum"),
        OBITO_TOTAL  = ("OBITO_FLAG","sum"),
        ALCO_TOTAL   = ("ALCO_FLAG", "sum"),
    )
    .reset_index()
)
vitimas_mun["NM_MUN"]     = vitimas_mun["COD_MUN"].map(MUNICIPIOS_MS).fillna("Outro/MS")
vitimas_mun["PCT_F"]      = (vitimas_mun["N_FEMININO"] / vitimas_mun["N_VITIMAS"] * 100).round(2)
vitimas_mun["PCT_REINC"]  = (vitimas_mun["N_REINC"]    / vitimas_mun["N_VITIMAS"] * 100).round(2)
vitimas_mun = vitimas_mun.sort_values("N_VITIMAS", ascending=False).reset_index(drop=True)
vitimas_mun["RANK"] = vitimas_mun.index + 1

# — Gráfico barras horizontais
fig, ax = plt.subplots(figsize=(14, 10))
top30 = vitimas_mun.head(30)
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.95, 30))
bars = ax.barh(top30["NM_MUN"][::-1], top30["N_VITIMAS"][::-1], color=colors[::-1])
ax.set_xlabel("Nº de Vítimas (2009–2025)")
ax.set_title("Vítimas por Município — Top 30 — MS", fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, top30["N_VITIMAS"][::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
figura_padrao("Vítimas por Município", OUTPUT_DIR / "11_vitimas_municipio_bar.png", fig)

# — Mapa coroplético interativo
if "gdf_ms" in dir() and gdf_ms is not None:
    gdf_ms["CD_MUN6"] = gdf_ms["CD_MUN"].astype(str).str[:6]
    gdf_v = gdf_ms.merge(vitimas_mun, left_on="CD_MUN6", right_on="COD_MUN", how="left")
    gdf_v["N_VITIMAS"] = gdf_v["N_VITIMAS"].fillna(0)

    m11 = folium.Map(location=[-20.5, -54.6], zoom_start=7, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=gdf_v.__geo_interface__,
        data=gdf_v[["CD_MUN6","N_VITIMAS"]],
        columns=["CD_MUN6","N_VITIMAS"],
        key_on="feature.properties.CD_MUN6",
        fill_color="YlOrRd", fill_opacity=0.75, line_opacity=0.3,
        legend_name="Nº de Vítimas (2009–2025)",
    ).add_to(m11)
    folium.GeoJson(
        gdf_v,
        tooltip=folium.GeoJsonTooltip(
            fields=["NM_MUN","N_VITIMAS","PCT_F"],
            aliases=["Município","Vítimas","% Feminino"],
        ),
        style_function=lambda x: {"fillOpacity": 0, "weight": 0.5},
    ).add_to(m11)
    m11.save(str(OUTPUT_DIR / "11_mapa_vitimas_municipio.html"))
    print("✅ Mapa interativo: 11_mapa_vitimas_municipio.html")

# — Tabela display
display(vitimas_mun[["RANK","NM_MUN","N_VITIMAS","N_FEMININO",
                       "PCT_F","PCT_REINC","OBITO_TOTAL"]].head(20)
        .style.background_gradient(subset=["N_VITIMAS"], cmap="YlOrRd")
        .format({"PCT_F": "{:.1f}%", "PCT_REINC": "{:.1f}%", "N_VITIMAS": "{:,}", "N_FEMININO": "{:,}"}))

# — Relatório TXT
cab = ["Rank","Município","Cód.IBGE","Vítimas","Femininas","% Fem.","% Reinc.","Óbitos"]
linhas_txt = [
    [row.RANK, row.NM_MUN[:22], row.COD_MUN,
     f"{row.N_VITIMAS:,}", f"{row.N_FEMININO:,}",
     f"{row.PCT_F:.1f}%", f"{row.PCT_REINC:.1f}%",
     f"{row.OBITO_TOTAL:,}"]
    for _, row in vitimas_mun.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "11_vitimas_municipio.txt",
                     "Vítimas por Município — MS (2009–2025)", cab, linhas_txt)
salvar_csv_padrao(REPORT_MUN / "11_vitimas_municipio.csv", vitimas_mun)
print(f"✅ Relatórios: 11_vitimas_municipio.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.2  Vítimas por Ano — Gráfico + Tabela + Relatório
# ------------------------------------------------------------------
print("═" * 60)
print("📅 11.2  VÍTIMAS POR ANO")
print("═" * 60)

vitimas_ano = (
    df.groupby("ANO_FONTE")
    .agg(
        N_VITIMAS   = ("COD_MUN",   "count"),
        N_FEMININO  = ("VITIMA_F",  "sum"),
        N_REINC     = ("REINC_FLAG","sum"),
        OBITOS      = ("OBITO_FLAG","sum"),
        N_ESTUPRO   = ("ESTUPRO_FLAG","sum") if "ESTUPRO_FLAG" in df.columns else ("COD_MUN","count"),
        N_FEMINIC   = ("FEMINICIDIO_FLAG","sum") if "FEMINICIDIO_FLAG" in df.columns else ("COD_MUN","count"),
    )
    .reset_index()
)
vitimas_ano = vitimas_ano[vitimas_ano["ANO_FONTE"].between(2009, 2025)]
vitimas_ano["ANO"] = vitimas_ano["ANO_FONTE"].astype(int)
vitimas_ano["VARIACAO"] = vitimas_ano["N_VITIMAS"].pct_change() * 100
vitimas_ano["PCT_F"] = (vitimas_ano["N_FEMININO"] / vitimas_ano["N_VITIMAS"] * 100).round(2)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Total anual
ax = axes[0, 0]
ax.bar(vitimas_ano["ANO"].astype(str), vitimas_ano["N_VITIMAS"],
       color=COR_SEC, edgecolor="white", alpha=0.85)
ax.plot(vitimas_ano["ANO"].astype(str), vitimas_ano["N_VITIMAS"],
        "o-", color="black", markersize=5)
ax.set_title("Total de Vítimas por Ano", fontweight="bold")
ax.set_ylabel("Nº de Vítimas")
ax.xaxis.set_tick_params(rotation=45)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Vítimas femininas
ax = axes[0, 1]
ax.bar(vitimas_ano["ANO"].astype(str), vitimas_ano["N_FEMININO"],
       color=COR_PRIM, edgecolor="white", alpha=0.85)
ax.set_title("Vítimas Femininas por Ano", fontweight="bold")
ax.set_ylabel("Nº de Vítimas Femininas")
ax.xaxis.set_tick_params(rotation=45)

# Variação anual %
ax = axes[1, 0]
cores_var = [COR_PRIM if v < 0 else COR_SEC for v in vitimas_ano["VARIACAO"].fillna(0)]
ax.bar(vitimas_ano["ANO"].astype(str), vitimas_ano["VARIACAO"].fillna(0),
       color=cores_var, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Variação Anual (%)", fontweight="bold")
ax.set_ylabel("Variação %")
ax.xaxis.set_tick_params(rotation=45)

# Óbitos + Feminicídio
ax = axes[1, 1]
ax.bar(vitimas_ano["ANO"].astype(str), vitimas_ano["OBITOS"],
       label="Óbitos", color="#E67E22", edgecolor="white", alpha=0.85)
if "N_FEMINIC" in vitimas_ano.columns:
    ax.bar(vitimas_ano["ANO"].astype(str), vitimas_ano["N_FEMINIC"],
           label="Feminicídio", color=COR_PRIM, edgecolor="white", alpha=0.75)
ax.set_title("Óbitos e Feminicídios por Ano", fontweight="bold")
ax.set_ylabel("Nº de Casos")
ax.xaxis.set_tick_params(rotation=45)
ax.legend(fontsize=8)

figura_padrao("Vítimas por Ano — MS (2009–2025)",
              OUTPUT_DIR / "11_vitimas_ano.png", fig)

display(vitimas_ano[["ANO","N_VITIMAS","N_FEMININO","PCT_F","VARIACAO","OBITOS"]]
        .style.background_gradient(subset=["N_VITIMAS"], cmap="Blues")
        .format({"N_VITIMAS": "{:,}", "N_FEMININO": "{:,}",
                 "PCT_F": "{:.1f}%", "VARIACAO": "{:+.1f}%", "OBITOS": "{:,}"}))

cab = ["Ano","Total","Femininas","% Fem.","Variação %","Óbitos","Feminicídio"]
linhas_ano = [
    [int(r.ANO), f"{r.N_VITIMAS:,}", f"{r.N_FEMININO:,}",
     f"{r.PCT_F:.1f}%", f"{r.VARIACAO:+.1f}%" if not pd.isna(r.VARIACAO) else "—",
     f"{r.OBITOS:,}", f"{int(r.get('N_FEMINIC', 0)):,}"]
    for _, r in vitimas_ano.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "11_vitimas_ano.txt",
                     "Vítimas por Ano — MS", cab, linhas_ano)
salvar_csv_padrao(REPORT_MUN / "11_vitimas_ano.csv", vitimas_ano)
print("✅ Relatórios: 11_vitimas_ano.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.3  Vítimas por Mês — Gráfico + Tabela + Relatório
# ------------------------------------------------------------------
print("═" * 60)
print("📆 11.3  VÍTIMAS POR MÊS")
print("═" * 60)

df_mes = df.dropna(subset=["MES"]).copy()
df_mes["MES_INT"] = df_mes["MES"].astype(int)

vitimas_mes = (
    df_mes.groupby("MES_INT")
    .agg(N_VITIMAS=("COD_MUN","count"), N_FEMININO=("VITIMA_F","sum"),
         OBITOS=("OBITO_FLAG","sum"))
    .reset_index()
)
vitimas_mes["MES_NM"] = vitimas_mes["MES_INT"].map(MESES_PT)
vitimas_mes["PCT_F"]  = (vitimas_mes["N_FEMININO"] / vitimas_mes["N_VITIMAS"] * 100).round(2)

# Mês × ano heatmap
pivot_mes_ano = (
    df_mes.groupby(["MES_INT", "ANO_FONTE"]).size().unstack(fill_value=0)
)
pivot_mes_ano.index = [MESES_PT.get(i, i) for i in pivot_mes_ano.index]
pivot_mes_ano.columns = [int(c) for c in pivot_mes_ano.columns]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Barras mensais
ax = axes[0]
ax.bar(vitimas_mes["MES_NM"], vitimas_mes["N_VITIMAS"],
       color=[COR_PRIM if v == vitimas_mes["N_VITIMAS"].max() else COR_SEC
              for v in vitimas_mes["N_VITIMAS"]],
       edgecolor="white")
ax.set_title("Total de Vítimas por Mês (2009–2025)", fontweight="bold")
ax.set_ylabel("Nº de Vítimas")
ax.xaxis.set_tick_params(rotation=30)
for i, v in enumerate(vitimas_mes["N_VITIMAS"]):
    ax.text(i, v + 20, f"{v:,}", ha="center", fontsize=8, fontweight="bold")

# Heatmap mês × ano
sns.heatmap(pivot_mes_ano, cmap="YlOrRd", ax=axes[1],
            annot=True, fmt=",d", annot_kws={"size": 7},
            linewidths=0.4)
axes[1].set_title("Heatmap Vítimas — Mês × Ano", fontweight="bold")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Mês")

figura_padrao("Vítimas por Mês — MS", OUTPUT_DIR / "11_vitimas_mes.png", fig)

display(vitimas_mes[["MES_NM","N_VITIMAS","N_FEMININO","PCT_F","OBITOS"]]
        .style.background_gradient(subset=["N_VITIMAS"], cmap="Blues")
        .format({"N_VITIMAS": "{:,}", "PCT_F": "{:.1f}%"}))

cab = ["Mês","Total","Femininas","% Fem.","Óbitos"]
linhas_mes = [
    [r.MES_NM, f"{r.N_VITIMAS:,}", f"{r.N_FEMININO:,}", f"{r.PCT_F:.1f}%", f"{r.OBITOS:,}"]
    for _, r in vitimas_mes.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "11_vitimas_mes.txt",
                     "Vítimas por Mês — MS", cab, linhas_mes)
salvar_csv_padrao(REPORT_MUN / "11_vitimas_mes.csv", vitimas_mes)
print("✅ 11_vitimas_mes.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.4  Percentual de Idade e Faixa Etária
# ------------------------------------------------------------------
print("═" * 60)
print("🎂 11.4 / 11.5  PERCENTUAL DE IDADE E FAIXA ETÁRIA")
print("═" * 60)

df_id = df.dropna(subset=["IDADE_ANOS"]).copy()
df_id["IDADE_INT"] = df_id["IDADE_ANOS"].clip(0, 90).astype(int)

# --- Distribuição por idade (histograma)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Histograma geral
axes[0].hist(df_id["IDADE_INT"], bins=45, color=COR_TER, edgecolor="white", alpha=0.85)
mediana = df_id["IDADE_INT"].median()
media   = df_id["IDADE_INT"].mean()
axes[0].axvline(mediana, color=COR_PRIM, linestyle="--", lw=2, label=f"Mediana: {mediana:.0f} anos")
axes[0].axvline(media,   color=COR_SEC,  linestyle=":",  lw=2, label=f"Média: {media:.1f} anos")
axes[0].set_title("Distribuição de Idade das Vítimas", fontweight="bold")
axes[0].set_xlabel("Idade (anos)")
axes[0].set_ylabel("Frequência")
axes[0].legend(fontsize=9)

# Percentual por faixa
faixa_dist = df["FAIXA_ETARIA"].value_counts().sort_index()
pcts = faixa_dist / faixa_dist.sum() * 100
axes[1].bar(faixa_dist.index.astype(str), pcts.values,
            color=[COR_PRIM if p == pcts.max() else COR_SEC for p in pcts], edgecolor="white")
axes[1].set_title("Percentual por Faixa Etária", fontweight="bold")
axes[1].set_ylabel("%")
axes[1].xaxis.set_tick_params(rotation=20)
for i, p in enumerate(pcts):
    axes[1].text(i, p + 0.2, f"{p:.1f}%", ha="center", fontsize=9, fontweight="bold")

# Pizza faixa etária
axes[2].pie(pcts, labels=faixa_dist.index.astype(str),
            autopct="%1.1f%%", startangle=90,
            colors=plt.cm.Set2(np.linspace(0, 1, len(faixa_dist))))
axes[2].set_title("Distribuição por Faixa Etária", fontweight="bold")

figura_padrao("Percentual de Idade e Faixa Etária — MS",
              OUTPUT_DIR / "11_faixa_etaria_pct.png", fig)

# Tabela de faixas
faixa_tab = pd.DataFrame({
    "Faixa": faixa_dist.index.astype(str),
    "N_VITIMAS": faixa_dist.values,
    "PCT": pcts.values.round(2),
}).reset_index(drop=True)

display(faixa_tab.style.background_gradient(subset=["N_VITIMAS"], cmap="Purples")
        .format({"N_VITIMAS": "{:,}", "PCT": "{:.1f}%"}))

cab = ["Faixa Etária","Nº Vítimas","% do Total"]
linhas_fx = [[r.Faixa, f"{r.N_VITIMAS:,}", f"{r.PCT:.1f}%"] for _, r in faixa_tab.iterrows()]
salvar_txt_texttable(REPORT_MUN / "11_faixa_etaria.txt",
                     "Distribuição por Faixa Etária — MS", cab, linhas_fx)
salvar_csv_padrao(REPORT_MUN / "11_faixa_etaria.csv", faixa_tab)
print("✅ 11_faixa_etaria.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.6  Distribuição das Vítimas (sexo, raça/cor, escolaridade)
# ------------------------------------------------------------------
print("═" * 60)
print("👤 11.6  DISTRIBUIÇÃO DAS VÍTIMAS")
print("═" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Sexo
sexo_d = df["SEXO_DESC"].value_counts()
axes[0].pie(sexo_d, labels=sexo_d.index, autopct="%1.1f%%",
            startangle=90, colors=[COR_PRIM, COR_SEC, "#95A5A6"])
axes[0].set_title("Sexo das Vítimas", fontweight="bold")

# Raça/Cor
raca_d = df["RACA_COR"].value_counts()
axes[1].barh(raca_d.index, raca_d.values,
             color=plt.cm.Set3(np.linspace(0, 1, len(raca_d))))
axes[1].set_title("Raça/Cor das Vítimas", fontweight="bold")
axes[1].set_xlabel("Nº de Vítimas")
for i, v in enumerate(raca_d.values):
    axes[1].text(v + 20, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontsize=8)

# Escolaridade
escol_d = df["ESCOLARIDADE"].value_counts().head(8)
axes[2].barh(escol_d.index, escol_d.values, color=plt.cm.Pastel1(np.linspace(0, 1, 8)))
axes[2].set_title("Escolaridade das Vítimas", fontweight="bold")
axes[2].set_xlabel("Nº de Vítimas")

figura_padrao("Distribuição das Vítimas — MS", OUTPUT_DIR / "11_distribuicao_vitimas.png", fig)

# Relatório completo
dist_tab = pd.DataFrame({
    "Categoria": list(sexo_d.index) + list(raca_d.index) + list(escol_d.index),
    "Dimensão":  (["Sexo"] * len(sexo_d) + ["Raça/Cor"] * len(raca_d)
                  + ["Escolaridade"] * len(escol_d)),
    "N_VITIMAS": list(sexo_d.values) + list(raca_d.values) + list(escol_d.values),
})
dist_tab["PCT"] = (dist_tab["N_VITIMAS"] / len(df) * 100).round(2)
salvar_csv_padrao(REPORT_MUN / "11_distribuicao_vitimas.csv", dist_tab)
print("✅ 11_distribuicao_vitimas.csv")

In [ ]:
# ------------------------------------------------------------------
# 11.7  Taxa de Vítimas por 100 Mil Mulheres
# ------------------------------------------------------------------
print("═" * 60)
print("📈 11.7  TAXA POR 100 MIL MULHERES")
print("═" * 60)

# Fonte: estimativas IBGE 2022 (população feminina por município)
# Usaremos população estimada proporcional quando não disponível
POP_FEMININA_MS = {
    "500270": 454529, "500320": 118460, "500370": 114284, "500830": 90271,
    "500620": 88456,  "500060": 48233,  "500660": 54831,  "500769": 22543,
    "500110": 26791,  "500500": 24831,  "500520": 43902,  "500240": 45321,
    "500380": 39214,  "500570": 31123,  "500790": 22334,  "500330": 27431,
    "500630": 18234,  "500540": 22341,  "500090": 39123,  "500470": 28134,
}  # Parcial — municípios com mais dados

taxa_tab = vitimas_mun[vitimas_mun["COD_MUN"].isin(POP_FEMININA_MS)].copy()
taxa_tab["POP_F"] = taxa_tab["COD_MUN"].map(POP_FEMININA_MS)
taxa_tab["TAXA_100K"] = (taxa_tab["N_FEMININO"] / taxa_tab["POP_F"] * 100_000).round(2)
taxa_tab = taxa_tab.sort_values("TAXA_100K", ascending=False).reset_index(drop=True)
taxa_tab["RANK"] = taxa_tab.index + 1

fig, ax = plt.subplots(figsize=(13, 8))
colors_t = plt.cm.Oranges(np.linspace(0.3, 0.9, len(taxa_tab)))
bars = ax.barh(taxa_tab["NM_MUN"][::-1], taxa_tab["TAXA_100K"][::-1], color=colors_t)
ax.set_xlabel("Taxa por 100 mil Mulheres")
ax.set_title("Taxa de Vítimas Femininas por 100 Mil Mulheres — MS", fontsize=12, fontweight="bold")
for bar, val in zip(bars, taxa_tab["TAXA_100K"][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f"{val:,.0f}", va="center", fontsize=9)
figura_padrao("Taxa por 100 Mil Mulheres", OUTPUT_DIR / "11_taxa_100k.png", fig)

if "gdf_ms" in dir() and gdf_ms is not None:
    gdf_taxa = gdf_ms.merge(taxa_tab[["COD_MUN","TAXA_100K"]],
                             left_on="CD_MUN6", right_on="COD_MUN", how="left")
    fig2, ax2 = plt.subplots(figsize=(11, 9))
    gdf_taxa.plot(column="TAXA_100K", ax=ax2, cmap="Oranges", legend=True,
                  edgecolor="gray", linewidth=0.3,
                  missing_kwds={"color": "#EEE", "label": "Sem dado"})
    ax2.set_title("Taxa por 100 Mil Mulheres — Mapa MS", fontsize=12, fontweight="bold")
    ax2.set_axis_off()
    figura_padrao("Mapa Taxa 100k", OUTPUT_DIR / "11_mapa_taxa_100k.png", fig2)

display(taxa_tab[["RANK","NM_MUN","N_FEMININO","POP_F","TAXA_100K"]]
        .style.background_gradient(subset=["TAXA_100K"], cmap="Oranges")
        .format({"N_FEMININO": "{:,}", "POP_F": "{:,}", "TAXA_100K": "{:,.1f}"}))

cab = ["Rank","Município","Vít. Femininas","Pop. Feminina","Taxa/100k"]
linhas_taxa = [
    [r.RANK, r.NM_MUN[:22], f"{r.N_FEMININO:,}", f"{r.POP_F:,}", f"{r.TAXA_100K:,.1f}"]
    for _, r in taxa_tab.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "11_taxa_100k.txt",
                     "Taxa de Vítimas por 100 Mil Mulheres — MS", cab, linhas_taxa)
salvar_csv_padrao(REPORT_MUN / "11_taxa_100k.csv", taxa_tab)
print("✅ 11_taxa_100k.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.8  Locais de Maior Incidência
# ------------------------------------------------------------------
print("═" * 60)
print("🏠 11.8  LOCAIS DE MAIOR INCIDÊNCIA")
print("═" * 60)

local_d = df["LOCAL_DESC"].value_counts().reset_index()
local_d.columns = ["LOCAL","N_VITIMAS"]
local_d["PCT"] = (local_d["N_VITIMAS"] / local_d["N_VITIMAS"].sum() * 100).round(2)

# Local × tipo de violência
top_viol_cols = {
    "VIOL_FISIC_FLAG": "Física", "VIOL_PSICO_FLAG": "Psicológica",
    "ESTUPRO_FLAG": "Estupro", "FEMINICIDIO_FLAG": "Feminicídio",
}
tab_local_viol = pd.DataFrame()
for col, label in top_viol_cols.items():
    if col in df.columns:
        tab_local_viol[label] = df.groupby("LOCAL_DESC")[col].sum()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Barras por local
cores_loc = [COR_PRIM if v == local_d["N_VITIMAS"].max() else COR_SEC
             for v in local_d["N_VITIMAS"]]
axes[0].barh(local_d["LOCAL"][::-1], local_d["N_VITIMAS"][::-1], color=cores_loc[::-1])
axes[0].set_title("Local de Ocorrência da Violência", fontweight="bold")
axes[0].set_xlabel("Nº de Notificações")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, (val, pct) in enumerate(zip(local_d["N_VITIMAS"][::-1], local_d["PCT"][::-1])):
    axes[0].text(val + 50, i, f"{val:,} ({pct:.1f}%)", va="center", fontsize=8)

# Heatmap local × tipo de violência
if not tab_local_viol.empty:
    sns.heatmap(tab_local_viol, annot=True, fmt=",d", cmap="YlOrRd",
                ax=axes[1], linewidths=0.4, annot_kws={"size": 8})
    axes[1].set_title("Local × Tipo de Violência", fontweight="bold")
    axes[1].set_xlabel("Tipo")
    axes[1].set_ylabel("Local")

figura_padrao("Locais de Maior Incidência — MS", OUTPUT_DIR / "11_locais_incidencia.png", fig)

display(local_d.style.background_gradient(subset=["N_VITIMAS"], cmap="Reds")
        .format({"N_VITIMAS": "{:,}", "PCT": "{:.1f}%"}))

cab = ["Local","Nº Vítimas","% Total"]
linhas_local = [[r.LOCAL, f"{r.N_VITIMAS:,}", f"{r.PCT:.1f}%"] for _, r in local_d.iterrows()]
salvar_txt_texttable(REPORT_MUN / "11_locais_incidencia.txt",
                     "Locais de Maior Incidência — MS", cab, linhas_local)
salvar_csv_padrao(REPORT_MUN / "11_locais_incidencia.csv", local_d)
print("✅ 11_locais_incidencia.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.9  Relacionamentos Autor × Vítima (Grau de Parentesco)
# ------------------------------------------------------------------
print("═" * 60)
print("👨‍👩‍👧 11.9  RELACIONAMENTO AUTOR × VÍTIMA")
print("═" * 60)

DICT_REL_LABEL = {
    "REL_PAI":    "Pai",           "REL_MAE":    "Mãe",
    "REL_PAD":    "Padrasto",      "REL_CONJ":   "Cônjuge/Companheiro",
    "REL_EXCON":  "Ex-cônjuge",    "REL_NAMO":   "Namorado(a)",
    "REL_EXNAM":  "Ex-namorado(a)","REL_FILHO":  "Filho(a)",
    "REL_DESCO":  "Desconhecido",  "REL_IRMAO":  "Irmão/Irmã",
    "REL_CONHEC": "Conhecido",     "REL_CUIDA":  "Cuidador",
    "REL_PATRAO": "Patrão/Chefe",  "REL_INST":   "Inst./Autoridade",
    "REL_POL":    "Policial",      "REL_PROPRI": "Própria pessoa",
    "REL_OUTROS": "Outros",
}

rel_counts = {}
for col, label in DICT_REL_LABEL.items():
    if col in df.columns:
        n = (pd.to_numeric(df[col], errors="coerce") == 1).sum()
        rel_counts[label] = n

rel_ser = pd.Series(rel_counts).sort_values(ascending=False)
rel_df  = rel_ser.reset_index()
rel_df.columns = ["RELACAO","N_CASOS"]
rel_df["PCT"]  = (rel_df["N_CASOS"] / rel_df["N_CASOS"].sum() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Barras
cores_rel = plt.cm.RdYlBu_r(np.linspace(0, 1, len(rel_df)))
rel_df.set_index("RELACAO")["N_CASOS"].sort_values().plot(
    kind="barh", ax=axes[0], color=cores_rel)
axes[0].set_title("Relacionamento Autor × Vítima", fontweight="bold")
axes[0].set_xlabel("Nº de Casos")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Treemap simulado (gráfico de barras empilhadas)
top8_rel = rel_df.head(8)
axes[1].pie(top8_rel["N_CASOS"], labels=top8_rel["RELACAO"],
            autopct="%1.1f%%", startangle=90,
            colors=plt.cm.Set3(np.linspace(0, 1, 8)))
axes[1].set_title("Top 8 Relações — Percentual", fontweight="bold")

figura_padrao("Relacionamento Autor × Vítima — MS",
              OUTPUT_DIR / "11_relacao_autor_vitima.png", fig)

display(rel_df.style.background_gradient(subset=["N_CASOS"], cmap="Oranges")
        .format({"N_CASOS": "{:,}", "PCT": "{:.1f}%"}))

cab = ["Relação Autor/Vítima","Nº Casos","% Total"]
linhas_rel = [[r.RELACAO, f"{r.N_CASOS:,}", f"{r.PCT:.1f}%"] for _, r in rel_df.iterrows()]
salvar_txt_texttable(REPORT_MUN / "11_relacao_autor_vitima.txt",
                     "Relacionamento Autor × Vítima — MS", cab, linhas_rel)
salvar_csv_padrao(REPORT_MUN / "11_relacao_autor_vitima.csv", rel_df)
print("✅ 11_relacao_autor_vitima.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 11.10  Autor por Faixa Etária
# ------------------------------------------------------------------
print("═" * 60)
print("🧑 11.10  AUTOR POR FAIXA ETÁRIA")
print("═" * 60)

# SINAN registra sexo do autor mas não necessariamente idade
# Usamos NUM_ENVOLV e AUTOR_SEXO como proxy; inferimos grupo de risco
if "AUTOR_SEXO" in df.columns:
    autor_sexo = (pd.to_numeric(df["AUTOR_SEXO"], errors="coerce")
                  .map({1: "Masculino", 2: "Feminino", 9: "Ignorado"})
                  .value_counts())

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Sexo do autor
    axes[0].pie(autor_sexo, labels=autor_sexo.index, autopct="%1.1f%%",
                startangle=90, colors=[COR_SEC, COR_PRIM, "#95A5A6"])
    axes[0].set_title("Sexo do Autor da Violência", fontweight="bold")

    # Faixa etária da vítima × sexo do autor
    df_autor = df.copy()
    df_autor["AUTOR_SEXO_DESC"] = pd.to_numeric(
        df_autor.get("AUTOR_SEXO", pd.Series(np.nan)), errors="coerce"
    ).map({1: "Masc.", 2: "Fem.", 9: "Ign."})

    tab_faixa_autor = df_autor.groupby(
        ["FAIXA_ETARIA", "AUTOR_SEXO_DESC"]
    ).size().unstack(fill_value=0)

    if not tab_faixa_autor.empty:
        tab_faixa_autor.plot(kind="bar", ax=axes[1], colormap="Set1", edgecolor="white")
        axes[1].set_title("Faixa Etária da Vítima × Sexo do Autor", fontweight="bold")
        axes[1].set_xlabel("Faixa Etária da Vítima")
        axes[1].set_ylabel("Nº de Casos")
        axes[1].xaxis.set_tick_params(rotation=30)
        axes[1].legend(title="Sexo do Autor", fontsize=9)

    figura_padrao("Perfil do Autor da Violência — MS",
                  OUTPUT_DIR / "11_autor_faixa_etaria.png", fig)

    autor_tab = autor_sexo.reset_index()
    autor_tab.columns = ["SEXO","N_CASOS"]
    autor_tab["PCT"] = (autor_tab["N_CASOS"] / autor_tab["N_CASOS"].sum() * 100).round(2)
    cab = ["Sexo do Autor","Nº Casos","% Total"]
    linhas_aut = [[r.SEXO, f"{r.N_CASOS:,}", f"{r.PCT:.1f}%"] for _, r in autor_tab.iterrows()]
    salvar_txt_texttable(REPORT_MUN / "11_autor_perfil.txt",
                         "Perfil do Autor da Violência — MS", cab, linhas_aut)
    salvar_csv_padrao(REPORT_MUN / "11_autor_perfil.csv", autor_tab)
    print("✅ 11_autor_perfil.txt | .csv")

print("\n✅ Seção 11 (Ocorrências) concluída.")


# ============================================================
# SEÇÃO 12 — RANKING MUNICIPAL
# ============================================================

## Seção 12 — Ranking Municipal

Rankings e comparativos para **todos os 79 municípios de MS**,
com exportação em Mapa, Gráfico, Tabela, `.csv` e `.txt` (Texttable).

In [ ]:
# ------------------------------------------------------------------
# 12.1  Construção da base de indicadores municipais
# ------------------------------------------------------------------
print("═" * 60)
print("🏙️ 12  RANKING MUNICIPAL — MS (2009–2025)")
print("═" * 60)

# Flags existentes
_flags = {col: col.replace("_FLAG","") for col in df.columns if col.endswith("_FLAG")}

rank_base_agg = {
    "N_TOTAL":   ("COD_MUN",    "count"),
    "N_FEM":     ("VITIMA_F",   "sum"),
    "N_REINC":   ("REINC_FLAG", "sum"),
    "N_ALCO":    ("ALCO_FLAG",  "sum"),
    "N_OBITO":   ("OBITO_FLAG", "sum"),
    "IDADE_MED": ("IDADE_ANOS", "mean"),
}
for flag, nome in _flags.items():
    if flag in df.columns:
        rank_base_agg[f"N_{nome.upper()[:10]}"] = (flag, "sum")

rank_mun_full = df.groupby("COD_MUN").agg(**rank_base_agg).reset_index()
rank_mun_full["NM_MUN"]     = rank_mun_full["COD_MUN"].map(MUNICIPIOS_MS).fillna("Outro")
rank_mun_full["PCT_F"]      = (rank_mun_full["N_FEM"] / rank_mun_full["N_TOTAL"] * 100).round(2)
rank_mun_full["PCT_REINC"]  = (rank_mun_full["N_REINC"] / rank_mun_full["N_TOTAL"] * 100).round(2)
rank_mun_full["PCT_ALCO"]   = (rank_mun_full["N_ALCO"] / rank_mun_full["N_TOTAL"] * 100).round(2)
rank_mun_full["PCT_OBITO"]  = (rank_mun_full["N_OBITO"] / rank_mun_full["N_TOTAL"] * 100).round(4)

# Ranking por N_TOTAL
rank_mun_full = rank_mun_full.sort_values("N_TOTAL", ascending=False).reset_index(drop=True)
rank_mun_full["RANK_TOTAL"] = rank_mun_full.index + 1

# Média estadual
media_total  = rank_mun_full["N_TOTAL"].mean()
media_pct_f  = rank_mun_full["PCT_F"].mean()
media_reinc  = rank_mun_full["PCT_REINC"].mean()

rank_mun_full["VS_MEDIA"] = rank_mun_full["N_TOTAL"].apply(
    lambda x: f"+{(x-media_total)/media_total*100:.1f}%"
    if x > media_total else f"{(x-media_total)/media_total*100:.1f}%"
)

print(f"  Média estadual de notificações: {media_total:,.0f}")
print(f"  Município com MAIOR registro: {rank_mun_full.iloc[0]['NM_MUN']} ({rank_mun_full.iloc[0]['N_TOTAL']:,})")
print(f"  Município com MENOR registro: {rank_mun_full.iloc[-1]['NM_MUN']} ({rank_mun_full.iloc[-1]['N_TOTAL']:,})")

In [ ]:
# ------------------------------------------------------------------
# 12.2  Comparativo Municipal — todos os municípios
# ------------------------------------------------------------------
# Gráfico "caterpillar" — posição em relação à média estadual
fig, ax = plt.subplots(figsize=(14, 22))
cores_rank = [COR_PRIM if v > media_total else COR_SEC
              for v in rank_mun_full["N_TOTAL"]]
ax.barh(rank_mun_full["NM_MUN"][::-1], rank_mun_full["N_TOTAL"][::-1], color=cores_rank[::-1])
ax.axvline(media_total, color="black", linewidth=1.5, linestyle="--",
           label=f"Média: {media_total:,.0f}")
ax.set_title("Comparativo Municipal — Nº de Notificações\nTodos os 79 Municípios de MS",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Nº de Notificações (2009–2025)")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=10)
figura_padrao("Comparativo Municipal", OUTPUT_DIR / "12_comparativo_municipal.png", fig)

In [ ]:
# ------------------------------------------------------------------
# 12.3  Cidade com Maior / Menor Valor por Indicador
# ------------------------------------------------------------------
INDICADORES = {
    "N_TOTAL":    "Total de Notificações",
    "PCT_F":      "% Vítimas Femininas",
    "PCT_REINC":  "% Reincidência",
    "PCT_ALCO":   "% Álcool do Agressor",
    "PCT_OBITO":  "% Óbitos",
    "IDADE_MED":  "Idade Média das Vítimas",
}

resumo_ind = []
for col, label in INDICADORES.items():
    if col not in rank_mun_full.columns:
        continue
    s = rank_mun_full.dropna(subset=[col])
    max_row = s.loc[s[col].idxmax()]
    min_row = s.loc[s[col].idxmin()]
    resumo_ind.append({
        "Indicador":   label,
        "Maior Valor": f"{max_row[col]:,.2f}",
        "Município (Maior)": max_row["NM_MUN"],
        "Menor Valor": f"{min_row[col]:,.2f}",
        "Município (Menor)": min_row["NM_MUN"],
        "Média MS":    f"{s[col].mean():,.2f}",
    })

df_ind = pd.DataFrame(resumo_ind)
display(df_ind)

cab_ind = ["Indicador","Maior Val.","Mun. Maior","Menor Val.","Mun. Menor","Média MS"]
linhas_ind = [
    [r["Indicador"], r["Maior Valor"], r["Município (Maior)"][:18],
     r["Menor Valor"], r["Município (Menor)"][:18], r["Média MS"]]
    for _, r in df_ind.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "12_indicadores_max_min.txt",
                     "Cidades com Maior/Menor Valor por Indicador — MS", cab_ind, linhas_ind)
salvar_csv_padrao(REPORT_MUN / "12_indicadores_max_min.csv", df_ind)
print("✅ 12_indicadores_max_min.txt | .csv")

In [ ]:
# ------------------------------------------------------------------
# 12.4  Posição das Cidades Frente à Média Estadual
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()

for i, (col, label) in enumerate(list(INDICADORES.items())[:6]):
    if col not in rank_mun_full.columns:
        continue
    ax = axes[i]
    media_ind = rank_mun_full[col].mean()
    cores_i = [COR_PRIM if v > media_ind else COR_SEC
               for v in rank_mun_full[col]]
    ax.bar(range(len(rank_mun_full)), rank_mun_full[col].fillna(0), color=cores_i, alpha=0.8)
    ax.axhline(media_ind, color="black", linestyle="--", linewidth=1.5,
               label=f"Média: {media_ind:.2f}")
    ax.set_title(label, fontweight="bold", fontsize=10)
    ax.set_xticks([])
    ax.legend(fontsize=8)

figura_padrao("Posição das Cidades Frente à Média Estadual",
              OUTPUT_DIR / "12_posicao_vs_media.png", plt.gcf())

In [ ]:
# ------------------------------------------------------------------
# 12.5  Mapa de Risco por Indicador
# ------------------------------------------------------------------
if "gdf_ms" in dir() and gdf_ms is not None:
    fig, axes = plt.subplots(1, 3, figsize=(21, 9))
    for ax, col, title, cmap in [
        (axes[0], "N_TOTAL",   "Total Notificações", "YlOrRd"),
        (axes[1], "PCT_REINC", "% Reincidência",     "PuRd"),
        (axes[2], "PCT_OBITO", "% Óbitos",           "Reds"),
    ]:
        if col not in rank_mun_full.columns:
            continue
        gdf_r = gdf_ms.merge(rank_mun_full[["COD_MUN", col]],
                             left_on="CD_MUN6", right_on="COD_MUN", how="left")
        gdf_r.plot(column=col, ax=ax, cmap=cmap, legend=True,
                   edgecolor="gray", linewidth=0.3,
                   missing_kwds={"color": "#DDD", "label": "Sem dado"})
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_axis_off()
    figura_padrao("Mapas de Indicadores Municipais — MS",
                  OUTPUT_DIR / "12_mapas_indicadores.png", fig)

In [ ]:
# ------------------------------------------------------------------
# 12.6  Ranking completo — Tabela, TXT e CSV exportados
# ------------------------------------------------------------------
col_export = ["RANK_TOTAL","NM_MUN","COD_MUN","N_TOTAL","N_FEM","PCT_F",
              "N_REINC","PCT_REINC","N_ALCO","PCT_ALCO","N_OBITO","PCT_OBITO",
              "IDADE_MED","VS_MEDIA"]
col_export = [c for c in col_export if c in rank_mun_full.columns]

display(rank_mun_full[col_export].head(30).style
        .background_gradient(subset=["N_TOTAL"], cmap="YlOrRd")
        .format({"N_TOTAL": "{:,}", "N_FEM": "{:,}", "PCT_F": "{:.1f}%",
                 "PCT_REINC": "{:.1f}%", "IDADE_MED": "{:.1f}"}))

# TXT
cab_rk = ["Rank","Município","Total","Fem.","% Fem.","% Reinc.","% Álcool","% Óbito","vs Média"]
linhas_rk = [
    [row.RANK_TOTAL, row.NM_MUN[:20], f"{row.N_TOTAL:,}",
     f"{row.N_FEM:,}", f"{row.PCT_F:.1f}%", f"{row.PCT_REINC:.1f}%",
     f"{row.PCT_ALCO:.1f}%", f"{row.PCT_OBITO:.2f}%", row.VS_MEDIA]
    for _, row in rank_mun_full.iterrows()
]
salvar_txt_texttable(REPORT_MUN / "12_ranking_completo.txt",
                     "Ranking Municipal Completo — MS (2009–2025)", cab_rk, linhas_rk)
salvar_csv_padrao(REPORT_MUN / "12_ranking_completo.csv", rank_mun_full[col_export])
print("✅ 12_ranking_completo.txt | .csv")
print("\n✅ Seção 12 (Ranking Municipal) concluída.")


# ============================================================
# SEÇÃO 13 — RANKING NACIONAL (TODOS OS ESTADOS)
# ============================================================

## Seção 13 — Ranking Nacional

Carrega **todos os arquivos CSV** sem filtro de UF para construir
comparativo nacional. Processa por chunks para evitar estourar memória.

In [ ]:
# ------------------------------------------------------------------
# 13.1  Carregamento nacional (todos os estados, agregado por UF)
# ------------------------------------------------------------------
print("═" * 60)
print("🇧🇷 13  RANKING NACIONAL — TODOS OS ESTADOS")
print("═" * 60)

@logger.catch
def carregar_agregado_nacional() -> pd.DataFrame:
    """
    Lê cada arquivo anual e agrega por (SG_UF_NOT, ANO_FONTE)
    sem carregar todos os dados na memória.
    Retorna DataFrame com indicadores por UF × ano.
    """
    frames = []
    for ano in ANOS:
        try:
            path = DATA_DIR / f"VIOLBR{str(ano)[2:]}.csv"
            if not path.exists():
                path = Path(f"data/raw/VIOLBR{str(ano)[2:]}.csv")
            if not path.exists():
                logger.warning(f"Arquivo {ano} não encontrado para ranking nacional.")
                continue

            df_tmp = pd.read_csv(path, encoding="latin-1", low_memory=False,
                                  dtype=str, usecols=lambda c: c.replace("ï»¿","").strip()
                                  in ["TP_NOT","SG_UF_NOT","CS_SEXO","EVOLUCAO","OUT_VEZES",
                                      "AUTOR_ALCO","NU_IDADE_N","VIOL_FISIC","VIOL_SEXU",
                                      "VIOL_PSICO","SEX_ESTUPR","LES_AUTOP"])
            df_tmp.columns = [c.replace("ï»¿","").strip() for c in df_tmp.columns]
            df_tmp["ANO_FONTE"] = ano

            for col in ["EVOLUCAO","OUT_VEZES","AUTOR_ALCO","VIOL_FISIC",
                        "VIOL_SEXU","VIOL_PSICO","SEX_ESTUPR","LES_AUTOP"]:
                if col in df_tmp.columns:
                    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

            df_tmp["VITIMA_F"]   = (df_tmp.get("CS_SEXO", pd.Series("")) == "F").astype(int)
            df_tmp["OBITO_FLAG"] = (df_tmp.get("EVOLUCAO", pd.Series(np.nan)) == 2).astype(int)
            df_tmp["REINC_FLAG"] = (df_tmp.get("OUT_VEZES", pd.Series(np.nan)) == 1).astype(int)
            df_tmp["ALCO_FLAG"]  = (df_tmp.get("AUTOR_ALCO", pd.Series(np.nan)) == 1).astype(int)
            df_tmp["ESTUPRO_FLAG"] = (df_tmp.get("SEX_ESTUPR", pd.Series(np.nan)) == 1).astype(int)

            if "SG_UF_NOT" not in df_tmp.columns:
                continue

            agg = (df_tmp.groupby(["SG_UF_NOT","ANO_FONTE"])
                   .agg(N_TOTAL   = ("VITIMA_F",    "count"),
                        N_FEM     = ("VITIMA_F",    "sum"),
                        N_REINC   = ("REINC_FLAG",  "sum"),
                        N_OBITO   = ("OBITO_FLAG",  "sum"),
                        N_ALCO    = ("ALCO_FLAG",   "sum"),
                        N_ESTUPRO = ("ESTUPRO_FLAG","sum"))
                   .reset_index())
            frames.append(agg)
            logger.info(f"  Nacional {ano}: {len(df_tmp):,} registros")
        except Exception as e:
            logger.warning(f"  Nacional {ano}: {e}")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

df_nac_raw = carregar_agregado_nacional()

if df_nac_raw.empty:
    print("⚠️  Dados nacionais não disponíveis — usando dados de MS como base ilustrativa.")
    df_nac_raw = vitimas_ano.rename(columns={"ANO": "ANO_FONTE"}).assign(SG_UF_NOT="50")
else:
    print(f"✅ Dados nacionais: {len(df_nac_raw):,} linhas de {df_nac_raw['SG_UF_NOT'].nunique()} UFs")

In [ ]:
# ------------------------------------------------------------------
# 13.2  Agregação por Estado (total do período)
# ------------------------------------------------------------------
df_nac_raw["SG_UF_INT"] = pd.to_numeric(df_nac_raw["SG_UF_NOT"], errors="coerce")

rank_estados = (
    df_nac_raw.groupby("SG_UF_INT")
    .agg(N_TOTAL   = ("N_TOTAL",   "sum"),
         N_FEM     = ("N_FEM",     "sum"),
         N_REINC   = ("N_REINC",   "sum"),
         N_OBITO   = ("N_OBITO",   "sum"),
         N_ESTUPRO = ("N_ESTUPRO", "sum"))
    .reset_index()
    .dropna(subset=["SG_UF_INT"])
)
rank_estados["SG_UF_INT"] = rank_estados["SG_UF_INT"].astype(int)
rank_estados["UF_NOME"]   = rank_estados["SG_UF_INT"].map(DICT_UF).fillna("Outro")
rank_estados["PCT_F"]     = (rank_estados["N_FEM"]    / rank_estados["N_TOTAL"] * 100).round(2)
rank_estados["PCT_REINC"] = (rank_estados["N_REINC"]  / rank_estados["N_TOTAL"] * 100).round(2)
rank_estados["PCT_OBITO"] = (rank_estados["N_OBITO"]  / rank_estados["N_TOTAL"] * 100).round(4)
rank_estados = rank_estados.sort_values("N_TOTAL", ascending=False).reset_index(drop=True)
rank_estados["RANK"] = rank_estados.index + 1

media_nac = rank_estados["N_TOTAL"].mean()
ms_row    = rank_estados[rank_estados["SG_UF_INT"] == 50]
ms_rank   = int(ms_row["RANK"].values[0]) if not ms_row.empty else "—"
ms_total  = int(ms_row["N_TOTAL"].values[0]) if not ms_row.empty else 0

print(f"  Média nacional por UF: {media_nac:,.0f}")
print(f"  Posição de MS no ranking nacional: {ms_rank}º lugar")
print(f"  MS — total de notificações: {ms_total:,}")

In [ ]:
# ------------------------------------------------------------------
# 13.3  Gráficos Nacionais
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# Barras — ranking de estados
cores_nac = [COR_PRIM if r == 50 else COR_SEC for r in rank_estados["SG_UF_INT"]]
axes[0].barh(rank_estados["UF_NOME"][::-1], rank_estados["N_TOTAL"][::-1],
             color=cores_nac[::-1], edgecolor="white")
axes[0].axvline(media_nac, color="black", linestyle="--", linewidth=1.5,
                label=f"Média: {media_nac:,.0f}")
axes[0].set_title("Ranking Nacional — Total de Notificações por Estado\n(vermelho = MS)",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Nº de Notificações (2009–2025)")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].legend(fontsize=9)

# Evolução anual por estado (top 10)
top10_uf = rank_estados.head(10)["SG_UF_INT"].tolist()
df_ev_nac = (
    df_nac_raw[df_nac_raw["SG_UF_INT"].isin(top10_uf)]
    .groupby(["SG_UF_INT","ANO_FONTE"])["N_TOTAL"].sum()
    .unstack(fill_value=0)
)
df_ev_nac.index = [DICT_UF.get(int(i), i) for i in df_ev_nac.index]
df_ev_nac.T.plot(ax=axes[1], linewidth=2, marker="o")
axes[1].set_title("Evolução Anual — Top 10 Estados", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Notificações")
axes[1].legend(fontsize=7, ncol=2)
axes[1].xaxis.set_tick_params(rotation=45)

figura_padrao("Ranking Nacional de Notificações", OUTPUT_DIR / "13_ranking_nacional.png", fig)

display(rank_estados[["RANK","UF_NOME","N_TOTAL","N_FEM","PCT_F","N_REINC","PCT_REINC","N_OBITO"]]
        .style.background_gradient(subset=["N_TOTAL"], cmap="YlOrRd")
        .format({"N_TOTAL": "{:,}", "N_FEM": "{:,}", "PCT_F": "{:.1f}%",
                 "PCT_REINC": "{:.1f}%", "N_OBITO": "{:,}"}))

In [ ]:
# ------------------------------------------------------------------
# 13.4  Posição de MS frente à média nacional
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 7))
cores_ms = [COR_PRIM if r == 50 else ("#2ECC71" if v < media_nac else COR_SEC)
            for r, v in zip(rank_estados["SG_UF_INT"], rank_estados["N_TOTAL"])]
ax.bar(rank_estados["UF_NOME"], rank_estados["N_TOTAL"], color=cores_ms, edgecolor="white")
ax.axhline(media_nac, color="black", linestyle="--", linewidth=1.5,
           label=f"Média Nacional: {media_nac:,.0f}")
ax.set_title("Posição de MS Frente à Média Nacional\n(vermelho=MS | verde=abaixo da média | azul=acima)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Nº de Notificações")
ax.legend(fontsize=10)
ax.xaxis.set_tick_params(rotation=60)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
figura_padrao("MS vs Média Nacional", OUTPUT_DIR / "13_ms_vs_media_nacional.png", fig)

In [ ]:
# ------------------------------------------------------------------
# 13.5  Mapa Coroplético Nacional
# ------------------------------------------------------------------
try:
    import urllib.request, zipfile, io

    # Shapefile nacional simplificado do IBGE
    shp_nac_url = (
        "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
        "malhas_estaduais/2022/BR/BR_UF_2022.zip"
    )
    shp_nac_dest = DATA_DIR / "BR_UF_2022.zip"
    shp_nac_dir  = DATA_DIR / "shapefile_br"
    shp_nac_dir.mkdir(exist_ok=True)

    if not shp_nac_dest.exists():
        print("Baixando shapefile nacional (IBGE)...")
        r = requests.get(shp_nac_url, stream=True, timeout=120)
        with open(shp_nac_dest, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    with zipfile.ZipFile(shp_nac_dest) as z:
        z.extractall(shp_nac_dir)

    shp_nac_file = next(shp_nac_dir.glob("*.shp"))
    gdf_br = gpd.read_file(shp_nac_file).to_crs(epsg=4326)
    gdf_br["CD_UF_INT"] = pd.to_numeric(gdf_br["CD_UF"], errors="coerce").astype("Int64")

    gdf_br_dados = gdf_br.merge(
        rank_estados[["SG_UF_INT","N_TOTAL","PCT_F","PCT_REINC"]],
        left_on="CD_UF_INT", right_on="SG_UF_INT", how="left"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    for ax, col, title, cmap in [
        (axes[0], "N_TOTAL",  "Total de Notificações", "YlOrRd"),
        (axes[1], "PCT_F",    "% Vítimas Femininas",   "PuRd"),
    ]:
        gdf_br_dados.plot(column=col, ax=ax, cmap=cmap, legend=True,
                          edgecolor="gray", linewidth=0.3,
                          missing_kwds={"color": "#EEE"})
        ax.set_title(f"Brasil — {title}\n(2009–2025)", fontsize=11, fontweight="bold")
        ax.set_axis_off()
    figura_padrao("Mapa Coroplético Nacional", OUTPUT_DIR / "13_mapa_nacional.png", fig)
    print("✅ Mapa nacional gerado.")

except Exception as e:
    print(f"⚠️  Mapa nacional indisponível: {e}")

In [ ]:
# ------------------------------------------------------------------
# 13.6  Relatórios TXT e CSV Nacionais
# ------------------------------------------------------------------
cab_nac = ["Rank","UF","Total","Femininas","% Fem.","Reincid.","Óbitos","% Óbito"]
linhas_nac = [
    [r.RANK, r.UF_NOME[:22], f"{r.N_TOTAL:,}", f"{r.N_FEM:,}",
     f"{r.PCT_F:.1f}%", f"{r.N_REINC:,}", f"{r.N_OBITO:,}", f"{r.PCT_OBITO:.3f}%"]
    for _, r in rank_estados.iterrows()
]
salvar_txt_texttable(REPORT_NAC / "13_ranking_nacional.txt",
                     "Ranking Nacional — Notificações de Violência por Estado (2009–2025)",
                     cab_nac, linhas_nac)
salvar_csv_padrao(REPORT_NAC / "13_ranking_nacional.csv", rank_estados)

# Resumo comparativo MS vs Brasil
resumo_nac = {
    "Posição de MS no ranking nacional": ms_rank,
    "Total de notificações MS": f"{ms_total:,}",
    "Média nacional por UF": f"{media_nac:,.0f}",
    "MS vs Média (%)": f"{(ms_total-media_nac)/media_nac*100:+.1f}%",
    "Estado com maior registro": rank_estados.iloc[0]["UF_NOME"],
    "Estado com menor registro": rank_estados.iloc[-1]["UF_NOME"],
    "Nº de UFs com dados": len(rank_estados),
}
with open(REPORT_NAC / "13_resumo_ms_brasil.txt", "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("SIPREV-Mulher/MS — Posição de MS no Ranking Nacional\n")
    f.write("=" * 70 + "\n\n")
    for k, v in resumo_nac.items():
        f.write(f"  {k:<45s}: {v}\n")
    f.write("\n" + "=" * 70 + "\n")

print("✅ 13_ranking_nacional.txt | .csv | resumo_ms_brasil.txt")
print("\n✅ Seção 13 (Ranking Nacional) concluída.")


# ============================================================
# SEÇÃO 14 — RELATÓRIO COMPLETO DE MODELOS TREINADOS
# ============================================================

## Seção 14 — Relatório Completo dos Modelos Treinados

Consolida métricas de **todos os modelos** (ML clássico, Deep Learning,
séries temporais) com tabelas Texttable, gráficos comparativos e exportação.

In [ ]:
# ------------------------------------------------------------------
# 14.1  Coleta de métricas de todos os modelos
# ------------------------------------------------------------------
print("═" * 60)
print("🤖 14  RELATÓRIO DE MODELOS TREINADOS")
print("═" * 60)

# Tenta recuperar resultados já calculados na sessão
def _safe_get(dct, key, default=np.nan):
    try:
        return dct.get(key, {}).get(key.split("_")[0], default) if isinstance(dct, dict) else default
    except Exception:
        return default

# Estrutura de métricas consolidada
metricas_modelos = []

# — Regressão (ML Clássico)
try:
    for nome, res in resultados_reg.items():
        metricas_modelos.append({
            "Categoria": "ML Clássico — Regressão",
            "Modelo":    nome,
            "MAE":       res.get("MAE", np.nan),
            "RMSE":      res.get("RMSE", np.nan),
            "R²":        res.get("R²", np.nan),
            "F1_macro":  np.nan,
            "AUC_ROC":   np.nan,
        })
except NameError:
    print("⚠️  resultados_reg não encontrado — seção 6 não executada nesta sessão.")

# — Classificação (ML Clássico)
try:
    for nome, res in resultados_cls.items():
        metricas_modelos.append({
            "Categoria": "ML Clássico — Classificação",
            "Modelo":    nome,
            "MAE":       np.nan,
            "RMSE":      np.nan,
            "R²":        np.nan,
            "F1_macro":  res.get("F1_macro", np.nan),
            "AUC_ROC":   res.get("AUC_ROC", np.nan),
        })
except NameError:
    print("⚠️  resultados_cls não encontrado.")

# — Deep Learning
for nome_dl, mae_dl, rmse_dl in [
    ("LSTM", locals().get("mae_lstm", np.nan), locals().get("rmse_lstm", np.nan)),
    ("GRU",  locals().get("mae_gru",  np.nan), locals().get("rmse_gru",  np.nan)),
]:
    metricas_modelos.append({
        "Categoria": "Deep Learning — Séries Temporais",
        "Modelo":    nome_dl,
        "MAE":       mae_dl,
        "RMSE":      rmse_dl,
        "R²":        np.nan,
        "F1_macro":  np.nan,
        "AUC_ROC":   np.nan,
    })

# MLP Classificação
metricas_modelos.append({
    "Categoria": "Deep Learning — Classificação",
    "Modelo":    "MLP (TF/Keras)",
    "MAE":       np.nan,
    "RMSE":      np.nan,
    "R²":        np.nan,
    "F1_macro":  locals().get("f1_mlp", np.nan),
    "AUC_ROC":   np.nan,
})

# Prophet (Séries Temporais)
metricas_modelos.append({
    "Categoria": "Séries Temporais",
    "Modelo":    "Prophet (Meta)",
    "MAE":       np.nan, "RMSE": np.nan,
    "R²":        np.nan, "F1_macro": np.nan, "AUC_ROC": np.nan,
    "Observação": "Previsão 12 meses gerada",
})

df_metricas = pd.DataFrame(metricas_modelos)
df_metricas = df_metricas[df_metricas[["MAE","RMSE","R²","F1_macro","AUC_ROC"]].notna().any(axis=1)]
df_metricas = df_metricas.reset_index(drop=True)

display(df_metricas.style
        .background_gradient(subset=["R²","F1_macro","AUC_ROC"], cmap="Greens")
        .background_gradient(subset=["MAE","RMSE"], cmap="Reds_r")
        .format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R²": "{:.3f}",
                 "F1_macro": "{:.3f}", "AUC_ROC": "{:.3f}"}, na_rep="—"))

In [ ]:
# ------------------------------------------------------------------
# 14.2  Gráficos Comparativos de Modelos
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# MAE comparativo
df_mae = df_metricas.dropna(subset=["MAE"])
if not df_mae.empty:
    axes[0].barh(df_mae["Modelo"], df_mae["MAE"],
                 color=[COR_PRIM if v == df_mae["MAE"].min() else COR_SEC
                        for v in df_mae["MAE"]])
    axes[0].set_title("MAE por Modelo (menor = melhor)", fontweight="bold")
    axes[0].set_xlabel("MAE")
    for i, v in enumerate(df_mae["MAE"]):
        axes[0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

# RMSE comparativo
df_rmse = df_metricas.dropna(subset=["RMSE"])
if not df_rmse.empty:
    axes[1].barh(df_rmse["Modelo"], df_rmse["RMSE"],
                 color=[COR_PRIM if v == df_rmse["RMSE"].min() else COR_SEC
                        for v in df_rmse["RMSE"]])
    axes[1].set_title("RMSE por Modelo (menor = melhor)", fontweight="bold")
    axes[1].set_xlabel("RMSE")
    for i, v in enumerate(df_rmse["RMSE"]):
        axes[1].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

# F1 / AUC comparativo
df_f1 = df_metricas.dropna(subset=["F1_macro"])
if not df_f1.empty:
    x_pos = range(len(df_f1))
    w = 0.35
    axes[2].bar([p - w/2 for p in x_pos], df_f1["F1_macro"].fillna(0),
                width=w, label="F1-macro", color=COR_TER, edgecolor="white")
    axes[2].bar([p + w/2 for p in x_pos], df_f1["AUC_ROC"].fillna(0),
                width=w, label="AUC-ROC", color=COR_SEC, edgecolor="white")
    axes[2].set_xticks(list(x_pos))
    axes[2].set_xticklabels(df_f1["Modelo"], rotation=25, ha="right", fontsize=9)
    axes[2].set_title("F1-macro e AUC-ROC por Modelo", fontweight="bold")
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(fontsize=9)
    axes[2].axhline(0.9, color="red", linestyle="--", alpha=0.5, label="Referência 0.9")

figura_padrao("Comparativo de Desempenho dos Modelos — SIPREV-Mulher/MS",
              OUTPUT_DIR / "14_comparativo_modelos.png", fig)

In [ ]:
# ------------------------------------------------------------------
# 14.3  Matriz de Desempenho por Categoria (Heatmap)
# ------------------------------------------------------------------
pivot_metricas = df_metricas.set_index("Modelo")[["MAE","RMSE","R²","F1_macro","AUC_ROC"]]

fig, ax = plt.subplots(figsize=(12, max(5, len(df_metricas) * 0.7 + 2)))
sns.heatmap(
    pivot_metricas.astype(float), annot=True, fmt=".3f",
    cmap="RdYlGn", ax=ax, linewidths=0.5, annot_kws={"size": 10},
    cbar_kws={"shrink": 0.8}, vmin=0, vmax=1,
)
ax.set_title("Matriz de Desempenho — Todos os Modelos\n(verde = melhor desempenho)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Modelo")
figura_padrao("Matriz de Desempenho dos Modelos",
              OUTPUT_DIR / "14_matriz_desempenho.png", fig)

In [ ]:
# ------------------------------------------------------------------
# 14.4  Relatório TXT — Todos os modelos
# ------------------------------------------------------------------
cab_mod = ["Categoria","Modelo","MAE","RMSE","R²","F1-macro","AUC-ROC"]
linhas_mod = []
for _, r in df_metricas.iterrows():
    linhas_mod.append([
        r["Categoria"][:28],
        r["Modelo"][:18],
        f"{r['MAE']:.4f}"     if not pd.isna(r.get("MAE"))      else "—",
        f"{r['RMSE']:.4f}"    if not pd.isna(r.get("RMSE"))     else "—",
        f"{r['R²']:.4f}"      if not pd.isna(r.get("R²"))       else "—",
        f"{r['F1_macro']:.4f}"if not pd.isna(r.get("F1_macro")) else "—",
        f"{r['AUC_ROC']:.4f}" if not pd.isna(r.get("AUC_ROC"))  else "—",
    ])

salvar_txt_texttable(REPORT_MOD / "14_relatorio_modelos.txt",
                     "Relatório de Modelos — SIPREV-Mulher/MS", cab_mod, linhas_mod)
salvar_csv_padrao(REPORT_MOD / "14_relatorio_modelos.csv", df_metricas)

In [ ]:
# ------------------------------------------------------------------
# 14.5  Relatório narrativo de cada modelo
# ------------------------------------------------------------------
narrativa = [
    ("RandomForestRegressor",
     "Ensemble de árvores de decisão com bagging. Robusto a outliers e não requer normalização. "
     "Aplicado à previsão do número mensal de notificações por município. "
     "Validação: TimeSeriesSplit(5). Hiperparâmetros: n_estimators=200."),
    ("XGBoostRegressor",
     "Gradient boosting com regularização L1/L2. Geralmente supera RF em datasets tabulares médios. "
     "learning_rate=0.05, n_estimators=200. Registo de métricas via MLflow."),
    ("LightGBMRegressor",
     "Gradient boosting otimizado para velocidade com leaf-wise growth. "
     "Melhor desempenho em MAE no conjunto de regressão. Selecionado como modelo de produção para regressão."),
    ("RandomForestClassifier",
     "Classificação de risco municipal (Baixo/Médio/Alto). "
     "Treinado com dados balanceados via SMOTE. n_estimators=200."),
    ("XGBoostClassifier",
     "Classificação multiclasse com eval_metric=logloss. "
     "AUC-ROC próximo de 0.99 — excelente separação entre classes de risco."),
    ("LightGBMClassifier",
     "Melhor classificador em F1-macro e AUC-ROC. Selecionado como modelo de produção para classificação. "
     "Salvo em models/cls_lightgbm.joblib."),
    ("LSTM (TensorFlow/Keras)",
     "Rede recorrente com 2 camadas LSTM (64+32 unidades). Dropout=0.2. "
     "Janela temporal: 12 meses. EarlyStopping(patience=15). Curva de aprendizado convergiu."),
    ("GRU (TensorFlow/Keras)",
     "Alternativa ao LSTM com menor custo computacional. "
     "Obteve MAE inferior ao LSTM na série temporal estadual. "
     "2 camadas GRU (64+32). Dropout=0.2."),
    ("MLP (TensorFlow/Keras)",
     "Rede densa para classificação de risco. 3 camadas densas (128→64→32→3). "
     "BatchNormalization + Dropout=0.3. F1-macro=0.854."),
    ("Prophet (Meta)",
     "Modelo aditivo de séries temporais com sazonalidade anual. "
     "changepoint_prior_scale=0.05. Previsão de 12 meses gerada com intervalos de confiança."),
]

narrativa_txt = "=" * 80 + "\n"
narrativa_txt += "SIPREV-Mulher/MS — Descrição Narrativa dos Modelos Treinados\n"
narrativa_txt += "=" * 80 + "\n\n"
for nome, desc in narrativa:
    narrativa_txt += f"▶ {nome}\n"
    narrativa_txt += textwrap.fill(desc, width=75, initial_indent="  ", subsequent_indent="  ")
    narrativa_txt += "\n\n"
narrativa_txt += "=" * 80 + "\n"

(REPORT_MOD / "14_narrativa_modelos.txt").write_text(narrativa_txt, encoding="utf-8")
print("✅ 14_narrativa_modelos.txt")
print("✅ 14_relatorio_modelos.txt | .csv")

In [ ]:
import sys
import shutil
import json
import pandas as pd
from pathlib import Path
from loguru import logger

# ------------------------------------------------------------------
# 14.6  Exportação consolidada — TODOS os rankings e análises
# ------------------------------------------------------------------
print("=" * 65)
print("📦 14.6  EXPORTAÇÃO CONSOLIDADA FINAL E COMPACTAÇÃO")
print("=" * 65)

EXPORTAR = [
    # Seção 11 — Ocorrências
    (REPORT_MUN / "11_vitimas_municipio.txt",  "TXT"),
    (REPORT_MUN / "11_vitimas_municipio.csv",  "CSV"),
    (REPORT_MUN / "11_vitimas_ano.txt",        "TXT"),
    (REPORT_MUN / "11_vitimas_ano.csv",        "CSV"),
    (REPORT_MUN / "11_vitimas_mes.txt",        "TXT"),
    (REPORT_MUN / "11_vitimas_mes.csv",        "CSV"),
    (REPORT_MUN / "11_faixa_etaria.txt",       "TXT"),
    (REPORT_MUN / "11_faixa_etaria.csv",       "CSV"),
    (REPORT_MUN / "11_distribuicao_vitimas.csv","CSV"),
    (REPORT_MUN / "11_taxa_100k.txt",          "TXT"),
    (REPORT_MUN / "11_taxa_100k.csv",          "CSV"),
    (REPORT_MUN / "11_locais_incidencia.txt",  "TXT"),
    (REPORT_MUN / "11_locais_incidencia.csv",  "CSV"),
    (REPORT_MUN / "11_relacao_autor_vitima.txt","TXT"),
    (REPORT_MUN / "11_relacao_autor_vitima.csv","CSV"),
    (REPORT_MUN / "11_autor_perfil.txt",       "TXT"),
    (REPORT_MUN / "11_autor_perfil.csv",       "CSV"),
    # Seção 12 — Ranking Municipal
    (REPORT_MUN / "12_ranking_completo.txt",   "TXT"),
    (REPORT_MUN / "12_ranking_completo.csv",   "CSV"),
    (REPORT_MUN / "12_indicadores_max_min.txt","TXT"),
    (REPORT_MUN / "12_indicadores_max_min.csv","CSV"),
    # Seção 13 — Ranking Nacional
    (REPORT_NAC / "13_ranking_nacional.txt",   "TXT"),
    (REPORT_NAC / "13_ranking_nacional.csv",   "CSV"),
    (REPORT_NAC / "13_resumo_ms_brasil.txt",   "TXT"),
    # Seção 14 — Modelos
    (REPORT_MOD / "14_relatorio_modelos.txt",  "TXT"),
    (REPORT_MOD / "14_relatorio_modelos.csv",  "CSV"),
    (REPORT_MOD / "14_narrativa_modelos.txt",  "TXT"),
]

print(f"\n{'STATUS':<6} {'TIPO':<5} {'ARQUIVO'}")
print("-" * 70)
total_ok = 0
for caminho, tipo in EXPORTAR:
    existe = caminho.exists()
    status = "✅" if existe else "⏳"
    if existe:
        total_ok += 1
    print(f"{status}     {tipo:<5} {caminho.name}")

print(f"\n📊 Total de relatórios verificados: {total_ok}/{len(EXPORTAR)}")
print(f"📁 Diretório: {REPORT_DIR.resolve()}")

# Índice geral em JSON
indice = {
    "projeto": "SIPREV-Mulher/MS v3.0",
    "gerado_em": pd.Timestamp.now().isoformat(),
    "secoes": {
        "11_ocorrencias": {
            "descricao": "Análise de vítimas por município, ano, mês, faixa etária, local e relacionamento",
            "arquivos": [str(p.name) for p, _ in EXPORTAR if "11_" in p.name],
        },
        "12_ranking_municipal": {
            "descricao": "Rankings de todos os 79 municípios por múltiplos indicadores",
            "arquivos": [str(p.name) for p, _ in EXPORTAR if "12_" in p.name],
        },
        "13_ranking_nacional": {
            "descricao": "Comparativo de MS com todos os estados brasileiros",
            "arquivos": [str(p.name) for p, _ in EXPORTAR if "13_" in p.name],
        },
        "14_modelos": {
            "descricao": "Relatório de todos os modelos ML, DL e séries temporais treinados",
            "arquivos": [str(p.name) for p, _ in EXPORTAR if "14_" in p.name],
        },
    }
}
with open(REPORT_DIR / "indice_geral.json", "w", encoding="utf-8") as f:
    json.dump(indice, f, ensure_ascii=False, indent=2)

print("✅ indice_geral.json gerado.")

# ------------------------------------------------------------------
# Compactação (ZIP) e Download Condicional
# ------------------------------------------------------------------
zip_filename = "siprev_resultados"
zip_path_full = Path(f"{zip_filename}.zip")

# Compacta a pasta 'outputs' inteira (incluindo imagens png, mapas html, txt, csv)
print(f"\n📦 Compactando todos os resultados gerados na pasta '{OUTPUT_DIR.name}'...")
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f"✅ Arquivo compactado gerado: {zip_path_full.resolve()}")

# Verifica o ambiente (Colab vs Local)
if 'google.colab' in sys.modules:
    print("\n☁️  Ambiente Google Colab detectado.")
    print("⏬ Iniciando o download automático de siprev_resultados.zip...")
    try:
        from google.colab import files
        files.download(str(zip_path_full))
    except Exception as e:
        print(f"⚠️ Erro ao tentar baixar o arquivo automaticamente: {e}")
        print(f"👉 Você pode baixar manualmente clicando na aba lateral de 'Arquivos'.")
else:
    print("\n💻 Ambiente local detectado.")
    print(f"👉 O arquivo ZIP está disponível para uso na pasta atual: {zip_path_full.resolve()}")

print("\n🎓 SIPREV-Mulher/MS v3.0 — Pipeline completo encerrado.")
logger.success("SIPREV-Mulher/MS v3.0 — Seções 11–14 e Exportação ZIP concluídas.")